# **NOTES**

(mainly from the books)

<div class="alert alert-block alert-info">

### **Week 1**

Introdction to PyHPC
</div>

<div class="alert alert-block alert-info">

### **Week 2**

Python Bootcamp
</div>


<div class="alert alert-block alert-danger">

##### **2.1 Command line arguments**

(book notes)

</div>

Using command line arguments in Python:
```python
import sys

print('Script Name:', sys.argv[0])
for i in range(1, len(sys.argv)):
    print(f'Argument {i}:', sys.argv[i])

# Run the script with command line arguments:
# python my_script.py arg1 arg2

# Output:
# Script Name: my_script.py
# Argument 1: arg1
# Argument 2: arg2
```



With sys.argv we can customize our scripts each time we run them, without having to change the code.

<div class="alert alert-block alert-danger">

##### **2.2 Sets**

(book notes)

</div>

Sets are an unordered collection of unique elements (so no duplicates). Basic uses here include membership testing (testing if something contains a specific element) and eliminating duplicate entries. 

<div class="alert alert-block alert-danger">


##### **2.3 Lambda expressions**

(book notes)

</div>

Lambda keyword are used to create small anonymous functions (so like small and unnamed functions). They are restricted to a single expression that is automatically returned, meaning they cannot contain complex algorithms or multiple lines of code.

For instance:
```python
def make_incrementor(n):
    return lambda x: x + n

f = make_incrementor(42)
f(0)
````
Returns 42, since the lambda function adds 42 to the input x (which is 0 in this case).

Then:
```python
f(1)
```
Returns 43, since the lambda function adds 42 to the input x (which is

More examples:
```python
pairs = [(1, 'one'), (2, 'two'), (3, 'three'), (4, 'four')]
pairs.sort(key=lambda pair: pair[1])
pairs
```

Returns ```[(4, 'four'), (1, 'one'), (3, 'three'), (2, 'two')]``` since the sort method sorts the list of pairs based on the second element of each pair (the string) in alphabetical order.

<div class="alert alert-block alert-danger">


##### **2.4 List comprehensions vs generators - performance and memory**

</div>


**List comprehension** - builds the entire list in memory eagerly:
```python
# Allocates a full list of 100,000 integers in memory
result = [i * i for i in range(100_000)]
```

**Generator expression** - computes values on demand (lazy), holds only one value at a time:
```python
# Holds nothing in memory until iterated
result = (i * i for i in range(100_000))
# Only evaluates when consumed:
total = sum(i * i for i in range(100_000))
```

**Performance comparison (from book benchmarks):**

| Style | Time |
|---|---|
| Explicit `for` loop with `append` | 16.1 ms |
| List comprehension | 10.1 ms |
| Generator with `sum` | 12.4 ms |

List comprehensions are fastest for building lists. Generators win on memory when you do not need all values at once, or when you can short-circuit early.

**When generators matter for HPC:** reading large files line-by-line with `yield` means only one line is in memory at a time - critical when files are larger than RAM.

```python
def get_temperatures(filename):
    with open(filename) as f:
        for row in csv.reader(f):
            temperature = int(row[temp_col]) / 10
            yield temperature   # caller gets one value at a time

# This never loads the whole file into memory:
for temp in get_temperatures('bigfile.csv'):
    process(temp)
```

> KEY DISTINCTION: `list(generator)` materialises all values - you lose the memory advantage. Only do this if you need to iterate multiple times or need random access.



**Dict comprehensions:**
```python
# Slightly faster and more readable than a loop:
squares = {i: i*i for i in range(100)}
```

<div class="alert alert-block alert-danger">


##### **2.5 Python data structures and their complexity**

</div>

These appear in the course exercises and in exam multiple choice questions about "which operation is O(1)?":

| Data structure | Operation | Complexity | Notes |
|---|---|---|---|
| `list` | `append` | O(1) amortised | Occasional reallocation |
| `list` | `pop()` (end) | O(1) | |
| `list` | `pop(0)` (front) | O(N) | Shifts all elements |
| `list` | `insert(0, x)` | O(N) | Shifts all elements |
| `list` | `x in list` | O(N) | Linear scan |
| `dict` | `d[key]` get/set | O(1) average | Hash table |
| `dict` | `key in d` | O(1) average | Hash table |
| `set` | `x in set` | O(1) average | Hash table |
| `set` | `set.add(x)` | O(1) average | |
| `collections.deque` | append/pop both ends | O(1) | Use instead of list when inserting at front |

KEY EXAM TRAP: Using a `list` for membership testing (`if x in my_list`) inside a loop is O(N) per check. If N is large, convert to a `set` first - then each check is O(1). This is a common "how would you speed this up?" question.


<div class="alert alert-block alert-danger">


##### **2.6 Functions, `*args`, `**kwargs`, and default arguments**

</div>


```python
def f(a, b, c=10):         # c has a default value
    return a + b + c

f(1, 2)        # c=10, returns 13
f(1, 2, 3)    # c=3, returns 6
f(1, b=2)     # keyword argument, returns 13

def g(*args):              # variable positional args → tuple
    return sum(args)

def h(**kwargs):           # variable keyword args → dict
    for k, v in kwargs.items():
        print(k, v)
```

**Mutable default argument trap** (classic Python bug, sometimes appears in exam code-reading questions):
```python
# WRONG - the list is created once at function definition, shared across calls
def append_to(val, lst=[]):
    lst.append(val)
    return lst

append_to(1)   # [1]
append_to(2)   # [1, 2]  ← not [2]! Same list object reused

# FIX - use None as sentinel
def append_to(val, lst=None):
    if lst is None:
        lst = []
    lst.append(val)
    return lst
```

<div class="alert alert-block alert-danger">


##### **2.7 Classes - the basics for HPC context**

</div>


Week 2 covers enough OOP to understand code in the exercises. The key things the exam might reference:

```python
class Particle:
    def __init__(self, x, y, mass):   # constructor
        self.x = x
        self.y = y
        self.mass = mass

    def kinetic_energy(self, velocity):
        return 0.5 * self.mass * velocity ** 2

    def __repr__(self):    # string representation for debugging
        return f"Particle(x={self.x}, y={self.y})"

p = Particle(0.0, 1.0, 2.5)
print(p.kinetic_energy(3.0))   # 11.25
```

**Why classes matter for HPC:** storing simulation state in a Python class object makes it easy to pass around but **impossible to use inside `@njit`** (Numba cannot compile code that accesses Python object attributes). The fix is always to extract scalars/arrays from the object before calling the compiled function.


<div class="alert alert-block alert-danger">


##### **2.8 Basic NumPy (week 2 level)**

</div>


Week 2 introduces NumPy to the level needed for week 3 onwards. Key operations:

```python
import numpy as np

# Creation
a = np.array([1, 2, 3], dtype='float64')
b = np.zeros((3, 4))             # 3×4 array of zeros
c = np.ones((3, 4), dtype='int32')
d = np.arange(0, 10, 2)         # [0, 2, 4, 6, 8]
e = np.linspace(0, 1, 5)        # [0.0, 0.25, 0.5, 0.75, 1.0]
f = np.random.rand(3, 3)        # uniform random in [0, 1)

# Shape and dtype
a.shape       # (3,)
b.shape       # (3, 4)
a.dtype       # dtype('float64')
a.ndim        # 1
b.size        # 12 (total elements)
b.nbytes      # 96 (bytes: 12 × 8)

# Indexing
b[0, 1]       # element at row 0, col 1
b[0, :]       # row 0 (all columns)
b[:, 1]       # column 1 (all rows)
b[1:3, :]     # rows 1 and 2

# Arithmetic - all element-wise
a + a         # [2, 4, 6]
a * 2         # [2, 4, 6]
np.sqrt(a)    # element-wise sqrt

# Reductions
a.sum()       # sum all elements
a.mean()
a.max(), a.min()
b.sum(axis=0) # sum along rows → shape (4,)
b.sum(axis=1) # sum along cols → shape (3,)

# Reshape
b.reshape(12)       # 1D, same data
b.reshape(2, 6)     # new shape, same data
b.T                 # transpose

# Stacking
np.vstack([a, a])   # stack vertically → (2, 3)
np.hstack([a, a])   # stack horizontally → (6,)
```

**Views vs copies (critical for performance):**
```python
# Slicing creates a VIEW - modifying it modifies the original
row = b[0, :]
row[0] = 999   # b[0, 0] is now 999!

# To get an independent copy:
row = b[0, :].copy()

# Fancy indexing always creates a COPY:
b[[0, 1]]   # copy, not view
```

<div class="alert alert-block alert-danger">


##### **2.9 Basic file I/O**

</div>

```python
# Writing
with open('output.txt', 'w') as f:
    f.write('Hello world\n')
    f.write(f'Value: {42}\n')

# Reading all lines
with open('output.txt', 'r') as f:
    lines = f.readlines()   # list of strings, including '\n'

# Reading line by line (memory-efficient for large files)
with open('output.txt', 'r') as f:
    for line in f:
        line = line.strip()   # remove trailing '\n'
        process(line)

# Writing a NumPy array to binary
np.save('data.npy', arr)        # saves as .npy
arr = np.load('data.npy')       # loads it back

# Writing/reading text (slow for large arrays)
np.savetxt('data.txt', arr)
arr = np.loadtxt('data.txt')
```

> KEY NOTE: For large arrays in HPC, use `np.save`/`np.load` (binary, fast) or `np.memmap` (week 8) - never `np.savetxt`/`np.loadtxt` for anything performance-sensitive.

<div class="alert alert-block alert-danger">


##### **2.10 Command-line argument patterns (HPC context)**

</div>


You already have `sys.argv` in section 2.1. The HPC-specific pattern that appears repeatedly in exercises:

```python
import sys
import numpy as np

# Pattern used in all job array scripts:
# Python receives $LSB_JOBINDEX as sys.argv[1]
job_index = int(sys.argv[1])      # 1-based (LSF convention)
array_index = job_index - 1       # convert to 0-based for Python

# Pattern used when a script accepts optional arguments:
if len(sys.argv) > 1:
    n_samples = int(sys.argv[1])
else:
    n_samples = 1000   # default

# Pattern for passing a file path:
data_path = sys.argv[1]           # e.g. /dtu/projects/02613_2025/data/
```

> KEY RULE: `$LSB_JOBINDEX` starts at 1 (LSF is 1-indexed). Python lists/arrays are 0-indexed. Always subtract 1 when using the job index to select from a Python list. This is the most common off-by-one bug in job array scripts.


<div class="alert alert-block alert-info">

### **Week 3**

The Memory Hierarchy
</div>


<div class="alert alert-block alert-danger">

##### **3.1 Memory hierarchy**

</div>

If one CPU operation takes $1$ second to complete, then the memory hierarchy can be visualized as follows:

> Register: **instant**

> L1 Cache: **$4$ seconds**

> L2 Cache: **$10$ seconds**

> L3 Cache: **$1$ minute**

> RAM: **$3$ minutes and $20$ seconds**

> SSD: **27.8 hours**

<div class="alert alert-block alert-danger">

##### **3.2 Cache lines**

</div>

When the CPU needs a value from RAM, it grabs the whole **cache line**, which is 64 bytes (is 8 float64 values), and it then stores the entire chunk in the fast L1 cache, and this is called **spatial locality** (= the fact that if you need element number 4 in an array, you will likely need element number 5 as well, so it makes sense to grab the whole cache line that contains both elements).



EXAMPLE:

Summing an array, if we have:
$$
\begin{bmatrix} 4 & 9 & 1 & 2 & 3 & 7 & 1 & 8 & 2 & 0 & 4 & 6 & 7 & 1 & 3 \end{bmatrix}
$$

Then we access the first 8 elements (red is the first cache line, green is the current index):
$$
\begin{bmatrix} 
\color{green}{4} & \color{red}{9} & \color{red}{1} & \color{red}{2} & \color{red}{3} & \color{red}{7} & \color{red}{1} & \color{red}{8} & 2 & 0 & 4 & 6 & 7 & 1 & 3 
\end{bmatrix}
$$
This is with current index `x[0] = 4` in green, and we can sum all the way to `x[7] = 8` (in red) without needing to access RAM again, since all those values are in the same cache line. However, when we need to access `x[8] = 2`, there is a cache miss, so we then fetch line `[8-15]` from RAM, which contains the remaining values that we can sum (all within cache hit, already in L1 cache) without RAM access.

<div class="alert alert-block alert-danger">


##### **3.3 Shrinking rows**

</div>

Say we want to downsamlpe a large image in two ways, then there can be a $12 \times$ difference.

Version A (row loop (fast)):

```python
for i in range(n // 2):
    samll[i, :] = im[i*2, :] + 
    im[i*2 + 1, :]
```

Each iteration reads two **full rows**. Rows are contiguous in memory, so good spatial locality, so lots of cache hits.

> (contiguous memory means storing data or programs in a single, continuous block of system memory with no gaps or interruptions)

Versian B (column loop (slow)):

```python
for i in range(n // 2):
    small[i, i] = im[i, i*2] +
    im[:, i*2+1]
```

Each iteration reads two full columns. Columns jump across the memory, so every access is a cache miss, so this is like $12 \times$ slower than version A.

> KEY INSIGHT: NumPy stores arrays in row-major (C-order). Element `arr[i, j]` is at memory address `base + i*cols + j` so consecutive elements of a row sit next to each other in RAM, but consecutive elements of a column are *cols* floats apart.

<div class="alert alert-block alert-danger">

##### **3.4 Reading the performance plots**

</div>

Exercises have produced a plot of MFLOP/s versus array size in KB. It looks like a descending staircase, where each step corresponds to the array outgrowing the cache level.

<img src="SymPyBilleder/2026-05-21-12-05-34.png" width="550">

> NOTE: 32 KB (L1 size (XeonGold6162)), 1 MB (L2 size), 19.7 MB (L3 size).

**How do we calculate array size in KB?**

A `SIZExSIZE` float64 matrix uses `SIZE*SIZE*8` bytes. We divide by 1024 to get KB, so:
$$
64 \times 64 = \frac{64^2 \times 8}{1024} = 32 \text{ KB}
$$
This is exactly L1 size, which is why the first drop appears.

Furthermore, a 1D row vector of SIZE elements uses `SIZE*8` bytes, so this is why the row vector experiment can probe much larger sizes before hitting a cache boundary.


<div class="alert alert-block alert-danger">

##### **3.5 Why FLOP/s and not raw time?**

</div>

Larger arrays take longer simply because there is more work. FLOP/s normalizes for that: it measures how fast the hardware is actually computing, not how much work there was. A drop in FLOP/s means the CPU is spending time waiting for data, not computing.

> NOTE MFLOP stands for "millions of floating-point operations per second" and is a measure of computational performance, so FLOP means "floating-point operations" and "per second" indicates the rate at which these operations are performed.

<div class="alert alert-block alert-danger">

##### **3.6 More on how to calculate expected cahce miss drop-off size for a matrix**

</div>

The core fomula asks: **How many bytes does our array occupy, and which cache level does it fit in?**

$$
\text{N elements} = \text{cache bytes} / \text{bytes per element}
$$

Here, a `float64` value takes 8 bytes as bytes per element so:

> 1D array of $N$ elements is `N*8` bytes.

> 2D matrix of `SIZExSIZE` elements is `SIZE*SIZE*8` bytes.

The drop happens when the array first stops fitting. So for `SIZE*SIZE*8 / 1024 > 32`, we can solve via:
$$
\text{SIZE} > \sqrt{\frac{32 \times 1024}{8}} = 64
$$
So again, a $64 \times 64$ matrix is the first to just barely fit in L1, so $65 \times 65$ is the first to not fit (so spills into L2), which is where the first MFLOP/s cliff appears.

> KEY NOTE: Remember that float64/ int64 is 8 bytes, float32/ int32 is 4 bytes, and float16/ int16 is 2 bytes.

Here is a Python script for calculation:
```python
# In Python / NumPy:
import numpy as np

arr = np.zeros((202,), dtype='float64') # pick between 64, 32, and 16 bit floats
print(arr.nbytes) # 1.616 bytes
print(arr.nbytes / 1024) # 1.58 KB
```

Here is a table for **1D float64 vector** on Xoen Gold6126:

|**Cache**| **Size**| **Bytes** | **Drop-off $N$ elements** |
|---|---|---| ---|
|L1|$32$ KB| $32 \times 1024 = 32768$ | $N >  32768/8 = 4096$ |
|L2|$1$ MB| $1 \times 1024 \times 1024 = 1048576$ | $N > 1048576/8 = 131072$ |
|L3|$19.7$ MB| $19.7 \times 1024 \times 1024 = 20643584$ | $N > 20643584/8 = 2580448$ |

Here is a table for **2D float64 matrix** on Xoen Gold6126:

|**Cache**| **Drop-off $N$ elements** |
|---|---|
| L1 | $\text{SIZE} > \sqrt{4096} = 64$ |
| L2 | $\text{SIZE} > \sqrt{131072} = 362$ |
| L3 | $\text{SIZE} > \sqrt{2580448} = 1607$ |

<div class="alert alert-block alert-danger">

##### **3.7 Compresing data before storing (why it is faster)**

</div>

The bottle neck for this is disk bandwidth and not CPU speed.

**The logic:** if the time to compress + write compressed < time to write raw, we win, and we use less disk space as bonus. The reverse applies for reading: decompress on the fly if decompression is faster than reading the extra bytes from disk.

If it pays off or not depends on:

> **1. Storage type:** HDD is very slow so compression almost always wins, with NVMe SSD being fast so it is less clear.

> **2. Algorithm:** Iz4 and blosc are designed to be fast (low CPU cost), and gzip is slow but achieves higher compression ratios.

> **3. Data content:** All-zeros compress to almost nothing, random floats barely compress at all.

<div class="alert alert-block alert-danger">


##### **3.8 Matrix loop-order**

</div>

SLOW VERSION:

> If we loop via `i` (row) then `j` (columns) then `k` (elements):



```python
for i in range(N):
    for j in range(N):
        for k in range(N):
            C[i, j] += A[i, k] * B[k, j]
```



In pure math, this is illustrated as:

$$
\begin{bmatrix} 
\color{blue}{A_{00}} & \color{blue}{A_{01}} & \color{blue}{A_{02}} & \color{blue}{A_{03}} & \color{blue}{A_{04}} \\
A_{10} & A_{11} & A_{12} & A_{13} & A_{14} \\
A_{20} & A_{21} & A_{22} & A_{23} & A_{24} \\
A_{30} & A_{31} & A_{32} & A_{33} & A_{34} \\
A_{40} & A_{41} & A_{42} & A_{43} & A_{44} 
\end{bmatrix}
\times
\begin{bmatrix} 
\color{blue}{B_{00}} & B_{01} & B_{02} & B_{03} & B_{04} \\
\color{red}{B_{10}} & B_{11} & B_{12} & B_{13} & B_{14} \\
\color{red}{B_{20}} & B_{21} & B_{22} & B_{23} & B_{24} \\
\color{red}{B_{30}} & B_{31} & B_{32} & B_{33} & B_{34} \\
\color{red}{B_{40}} & B_{41} & B_{42} & B_{43} & B_{44} 
\end{bmatrix}
=
\begin{bmatrix} 
\color{blue}{C_{00}} & C_{01} & C_{02} & C_{03} & C_{04} \\
C_{10} & C_{11} & C_{12} & C_{13} & C_{14} \\
C_{20} & C_{21} & C_{22} & C_{23} & C_{24} \\
C_{30} & C_{31} & C_{32} & C_{33} & C_{34} \\
C_{40} & C_{41} & C_{42} & C_{43} & C_{44} 
\end{bmatrix}
$$


So above, we see that we move along the row of the mist matrix $A$ following the blue elements, and for each blue element in the row, we move one red element in the column of matrix $B$.

So `B[k,j]` steps down a column, so every $k$'th step is a cache miss on matrix $B$, and each miss fetches a new 64-byte (8 floats) cache line but uses only $1$ of those $8$ values is used before $k$ increments to the next row, and we fetch an entire new cache line.

When we have completed the $B$ column and first row of $A$, we can move to the next $C$ element, so we go from $C_{00}$ to $C_{01}$, and we repeat the same process.

> NOTE: first, when we have completed entire $C$ row of elements do we move on to the next row in $A$, were we repeat all column-wise accesses to $B$ for the next row of $A$.

> TO SUMMARIZE: `A[i,k]` with `i` is fixed, `k` increases = we step along row `i` of $A$ so row-major, perfectly sequential, cache friendly. But `B[k,j]` is where `j` is fixed, `k` increases, we step down column `j` of $B$ = each element is a full row-width apart in memory, and every single access is in a different cache line.


> If we loop via `i` (row) then `k` (elements) then `j` (columns):

> THIS IS FAST:

```python
for i in range(N):
    for k in range(N):
        for j in range(N):
            C[i, j] += A[i, k] * B[k, j]
```

$$
\begin{bmatrix} 
\color{blue}{A_{00}} & A_{01} & A_{02} & A_{03} & A_{04} \\
A_{10} & A_{11} & A_{12} & A_{13} & A_{14} \\
A_{20} & A_{21} & A_{22} & A_{23} & A_{24} \\
A_{30} & A_{31} & A_{32} & A_{33} & A_{34} \\
A_{40} & A_{41} & A_{42} & A_{43} & A_{44} 
\end{bmatrix}
\times
\begin{bmatrix} 
\color{blue}{B_{00}} & \color{blue}{B_{01}} & \color{blue}{B_{02}} & \color{blue}{B_{03}} & \color{blue}{B_{04}} \\
B_{10} & B_{11} & B_{12} & B_{13} & B_{14} \\
B_{20} & B_{21} & B_{22} & B_{23} & B_{24} \\
B_{30} & B_{31} & B_{32} & B_{33} & B_{34} \\
B_{40} & B_{41} & B_{42} & B_{43} & B_{44} 
\end{bmatrix}
=
\begin{bmatrix} 
\color{blue}{C_{00}} & \color{blue}{C_{01}} & \color{blue}{C_{02}} & \color{blue}{C_{03}} & \color{blue}{C_{04}} \\
C_{10} & C_{11} & C_{12} & C_{13} & C_{14} \\
C_{20} & C_{21} & C_{22} & C_{23} & C_{24} \\
C_{30} & C_{31} & C_{32} & C_{33} & C_{34} \\
C_{40} & C_{41} & C_{42} & C_{43} & C_{44} 
\end{bmatrix}
$$


Here, for each $A_{00}$ element, we access the entire first row of $B$ and first row of $C$, and when we have completed one round, we move to $A_{01}$ and move to the second row of $B$, but we stay in the first row of $C$.

Both `B[k,j]` and `C[i,j]` step along the rows so this is **sequential memory access**, and every cache line loaded is fully used, where `A[i,k]` is constant for the entire inner loop, so it stays in a register. We end up with $8$ useful opretation per cache miss instead of $1$.

> TO SUMMARIZE: `A[i,k]`with both `i` and `k` being fixed, this is the same value every iteration, so the compiler keeps it in a CPU register with zero memory accesses for $A$ in the inner loop. `B[k,j]` with `k` being fixed as `j` increases, so we step along row `k` of $B$ = sequential cache friendly. `C[i,j]` with `i` being fixed as `j` increases, so we step along row `i` of $C$ = sequential cache friendly.

<div class="alert alert-block alert-danger">

##### **3.9 Matrix multiplication**

</div>

To compute $C= A \times B$ every element `C[i,j]` is the dot product of row `i` of $A$ with column `j` of $B$:
$$
C[i,j] = \sum_k A[i,k] \cdot B[k,j]
$$
The inner sum over $k$ is where all the work happens, and also where the cache behaviour differs completely between the two loop ordering.

<div class="alert alert-block alert-danger">

##### **3.10 Performance measurement**

</div>


Before we can improve performance, we have to measure it. The lecture introduces three distinct metrics for `s = sum(x)`:

> **Wall-clock time** - how long we actually waited. Measured with `perf_counter`. This is the most honest measure because it includes everything: computation, memory waits, OS scheduling.

> **CPU time** - how much time the CPU spent working. For single-threaded code this is slightly less than wall time (the OS occasionally preempts your process). For parallel code, CPU time can be **larger** than wall time because it sums across all cores.

> **FLOP/s** - floating point operations per second. Measures how fast the hardware is actually computing, independent of how much work there was.

$$
\text{FLOP/s} = \frac{\text{number of FLOPs}}{\text{wall time (seconds)}}
$$

> CRITICAL NOTE: There is no automatic tool to count FLOPs, so we must look at the code and count manually.

```python
from time import perf_counter as time

t = time()
for _ in range(100):   # repeat to amortise Python overhead
    s = sum(x)
t = time() - t
t = t / 100            # average per call
```


<div class="alert alert-block alert-danger">

##### **3.11 How to count FLOPs from code**

</div>

This is a key exam skill. Every arithmetic operation on floating-point numbers ($+$, $-$, $*$, $/$) is one FLOP. Integer indexing, comparisons, and assignments are NOT FLOPs.

**Worked example** (from the 2024R exam):

```python
for i in range(n_steps):       # not a FLOP
    a = x[i] * x[i] + 4        # 2 FLOPs: one multiply, one add
    b = y[n - i - 1] / x[i]    # 1 FLOP: one divide
    z = z + a / b              # 2 FLOPs: one divide, one add
```

FLOPs per iteration: $2 + 1 + 2 =$ **5 FLOPs**

Total FLOPs for 10,000 iterations: $5  \times 10,000 =$ **50,000 FLOPs**

Total wall time: $2000 + 5000 + 5000 + 3000 = 15,000 \: \mu s =$ **0.015 seconds**

$$
\text{FLOP/s} = \frac{50{,}000}{0.015} = 3.33 \times 10^6 \text{ FLOP/s}
$$

> KEY RULE: Count only arithmetic on floats. Adding an integer index like `n - i - 1` where n and i are ints does NOT count. But if the array contains floats and you divide, that counts.

> NOTE: MFLOP/s = FLOP/s / 1,000,000. So $3.33 \times 10^6$ FLOP/s = 3.33 MFLOP/s.

<div class="alert alert-block alert-danger">

##### **3.12 Finding cache sizes with `lscpu`**

</div>

On the HPC node, run `lscpu` to get the actual cache sizes for that specific CPU:

```bash
$ lscpu
...
L1d cache:    32K
L2 cache:     1024K
L3 cache:     19712K
```

These are the numbers to use in all drop-off calculations. Never use the numbers from own laptop - the exercises must be run on the HPC node (the cache sizes differ).

**About cache size / MFLOP/s drop-off:**

Given a matrix of shape `(SIZE, SIZE)` dtype float64, at what SIZE does performance drop (cache miss)?

$$\text{SIZE} = \sqrt{\frac{\text{cache bytes}}{8}}$$

So for L1 = 32 KB:
$$\text{SIZE} = \sqrt{\frac{32 \times 1024}{8}} = \sqrt{4096} = 64$$

Performance drops when SIZE > 64 (array no longer fits in L1).

<div class="alert alert-block alert-danger">

##### **3.13 Takeaways (lecture summary)**

</div>

The three core messages from Week 3, stated explicitly on the final slide:

> **Hardware matters** - the same code runs at wildly different speeds depending on whether data fits in cache or not.

> **Performance depends on data, not just code** - the column loop and the row loop have identical FLOPs. The $12x$ difference is purely from access pattern.

> **Minimising data transfer = faster programs** - this applies at every level: fewer cache misses, less RAM access, less disk I/O, less PCIe transfer to GPU.

<div class="alert alert-block alert-info">

### **Week 4**

Profiling and High-Performance NumPy
</div>


<div class="alert alert-block alert-danger">

##### **4.1 The profiling workflow**

</div>


The golden rule: **never guess where the bottleneck is, we measure it**. The workflow is always two steps:

**Step 1: Function-level profiling** with `cProfile`. Low overhead, tells us *which function* is slow:

```python
python -m cProfile -s cumulative script.py   # print to terminal
python -m cProfile -o script.prof script.py  # save for snakeviz
snakeviz script.prof                         # open interactive GUI
```

The output columns to focus on:

> `tottime` is time spent *inside* this function, excluding subcalls. This is the real bottleneck indicator.

> `cumtime` is total time including all subcalls. Useful for finding which high-level function to investigate.

> `ncalls` is number of calls. A function called $1,000,000$ times with $1 \: \mu s$ each is just as bad as one called once taking $1$ second.

**Step 2: Line-level profiling** with `kernprof`. Much higher overhead but tells us *exactly which line* inside the slow function costs the most:

```python
@profile                                          # decorator on the function to profile
kernprof -l script.py                             # 1. run profiling, writes script.py.lprof
python -m line_profiler -rmt "script.py.lprof"    # 2. show results
```

> NOTE: The `@profile` decorator only works when running under `kernprof`. It does not exist in normal Python, so remove it (or wrap in try/except) for regular runs.

> KEY INSIGHT: Always start with `cProfile` to narrow the search space, then use `line_profiler` only on the confirmed bottleneck. Line profiling everything is prohibitively slow.


<div class="alert alert-block alert-danger">

##### **4.2 Stride formula**

</div>

**The formula is given as:**

For a **C-order (row-major)** array, strides are calculated from **right to left**:

> The **last dimension** always has stride = `bytes_per_element` (dtype size)

> Each dimension to the left = product of all dimensions to its right $ \times $ bytes_per_element

In other words:

$$\text{stride}[i] = \left(\prod_{j=i+1}^{n} \text{shape}[j]\right) \times \text{itemsize}$$

**Example 1: `shape (2, 3)`, int32 (4 bytes):**

Start from the right:

> **Last dim (axis 1):** stride = `4` bytes (one int32)

> **Second-to-last (axis 0):** stride = `3  x 4 = 12` bytes (jump over a full row of 3 elements)

SO strides = `(12, 4)` ✓

> Intuitively: to move one step along axis 0 (go to the next row), you skip over 3 elements, each 4 bytes = 12 bytes.

**Example 2: `shape (32, 3, 64, 64)`, float64 (8 bytes):**

Start from the right:

| Axis | Calculation | Stride |
|---|---|---|
| 3 (last) | `8` | `8` |
| 2 | `64  x 8` | `512` |
| 1 | `64  x 64  x 8` | `32,768` |
| 0 | `3  x 64  x 64  x 8` | `98,304` |

SO strides = `(98304, 32768, 512, 8)` ✓

**QUICK MENTAL NOTE:**

Always start from the right with `itemsize`, then multiply by each successive dimension as we move left.

We can also just check in Python:

```python
a = np.zeros((32, 3, 64, 64), dtype='float64')
print(a.strides)  # (98304, 32768, 512, 8)
print(a.dtype.itemsize)  # 8
```

> NOTE: This only holds for **C-order** (the default). Fortran-order (`order='F'`) goes the opposite direction. strides are built left-to-right, which is why a Fortran array reshaped to C-order needs a copy. The stride pattern cannot be reinterpreted without reordering the bytes.

The stride directly tells us the cache cost of looping over a dimension:

> Small stride (e.g. 8) → consecutive elements are adjacent in memory → 
  cache hits → put this dimension in the **innermost loop**

> Large stride (e.g. 512) → consecutive elements are far apart in memory → 
  cache misses → put this dimension in the **outermost loop**

**General rule: sort dimensions by stride, smallest innermost.**

For any C-order (row-major) NumPy array, the last index always has the 
smallest stride, so the last index always goes in the innermost loop. If 
you are given explicit strides, just find the smallest one and loop over 
that dimension innermost.

Example: strides = `(512, 64, 8)` → loop order should be dim 0 outer, 
dim 1 middle, dim 2 inner:
```python
for i in range(shape[0]):      # stride 512 - outermost
    for j in range(shape[1]):  # stride 64  - middle
        for k in range(shape[2]):  # stride 8 - innermost
            ... a[i, j, k] ...
```

<div class="alert alert-block alert-danger">

##### **4.3 What is a NumPy array (under the hood)**

</div>

A NumPy array is not just data, it is a metadata object pointing to a flat block of memory. The three key fields are:

> `shape` is a tuple giving the number of elements along each dimension, e.g. `(2, 3)`.

> `dtype` is the data type, e.g. `float64` (8 bytes per element).

> `strides` is a tuple giving **how many bytes to jump** in memory to move one step along each dimension.

EXAMPLE:

We have a `(2, 3)` int32 array has strides `(12, 4)`. To get from row $0$ to row $1$, jump 12 bytes ($= 3 \: \text{elements} \times 4 \: \text{bytes}$). To get from column $0$ to column $1$, jump 4 bytes ($= 1 \: \text{element} \times 4 \: \text{bytes}$).

```python
a = np.array([[1, 2, 3], [4, 5, 6]])
# shape: (2, 3),  strides: (12, 4),  dtype: int32
```

To find element `a[i, j]` in memory:
$$
\text{byte address} = \text{base} + i \times \text{strides}[0] + j \times \text{strides}[1]
$$

For a 4D array `np.zeros((32, 3, 64, 64))` with float64:

> strides = `(98304, 32768, 512, 8)`

> Moving along the last axis: 8 bytes (one float64) is cache friendly.

> Moving along the first axis: 98304 bytes this jumps across a huge chunk of memory.

> KEY NOTE: `arr.nbytes` gives total bytes. `arr.size * arr.dtype.itemsize` is equivalent.


<div class="alert alert-block alert-danger">


##### **4.4 Views vs copies**

</div>

This is one of the most important and most misunderstood parts of NumPy.

A **view** is a new array object that *shares the same underlying memory* as the original. No data is copied. Modifying the view modifies the original.

A **copy** allocates new memory and duplicates the data. Independent of the original.

```python
a = np.array([[1, 2, 3], [4, 5, 6]])

b = a.T          # VIEW - transpose just swaps strides: (4, 12) instead of (12, 4)
b = a.reshape(3, 2)   # VIEW -  same data, different shape/strides
b = a[1, :]      # VIEW -  row slice, stride (4,)
b = a[:, 1]      # VIEW -  column slice, stride (12,) -  still a view!
b = a[:, ::-1]   # VIEW -  reversed columns, negative stride (-4,)

b = a[[0, 1], [0, 2]]  # COPY -  fancy indexing always makes a copy
b = a.copy()           # COPY -  explicit
```

To check: `np.shares_memory(a, b)` returns `True` if they share memory, so it is a **view**.
    
> KEY INSIGHT: Fortran-order arrays (`order='F'`) store columns contiguously instead of rows. Reshaping a Fortran array forces a **copy** because the stride pattern cannot represent the new shape without reordering data. This is the most common hidden copy in NumPy.

> RULE: Views save both memory and compute time. Prefer them whenever possible. The cost of an unintended copy on a large array can dominate your runtime.


<div class="alert alert-block alert-danger">

##### **4.5 Broadcasting**

</div>

Broadcasting is NumPy's mechanism for performing element-wise operations on arrays with *different shapes*, without copying data. The rule is applied dimension-by-dimension from the right:

**The 4 broadcasting rules:**
1. Line up shapes to the right (right-align the tuples)
2. Left-pad the shorter shape with 1s
3. Any dimension of size 1 is "stretched" to match the other
4. If all dimensions match after stretching: compatible. Otherwise: `ValueError`.

```python
a.shape = (2, 3)
b.shape =   (3,)   →  padded to (1, 3)  →  stretched to (2, 3)  ✓

a.shape = (2, 3)
b.shape =   (2,)   →  padded to (1, 2)  →  can't stretch: 2 ≠ 3  ✗
```

**The `None` / `np.newaxis` trick**, manually add a size-1 dimension to control how broadcasting aligns:

```python
x = np.array([1, 2, 3])   # shape (3,)
y = np.array([10, 20])    # shape (2,)

# Outer product -  all combinations:
x[:, None] * y[None, :]
# (3, 1) * (1, 2) → (3, 2)  ✓

# Pairwise absolute differences:
np.abs(x[:, None] - y[None, :])
# (3, 1) - (1, 2) → (3, 2)
```

> KEY INSIGHT: `x[:, None]` adds a new axis at position 1, making $x$ a column vector of shape `(N, 1)`. `y[None, :]` adds a new axis at position $0$, making $y$ a row vector of shape `(1, M)`. Their broadcast product computes all $N\times M$ combinations in a single C-level operation so no Python loop.


<img src="SymPyBilleder/2026-05-21-21-59-02.png" width="550">


<div class="alert alert-block alert-danger">

##### **4.6 Broadcasting worked example - exam style (4D case)**

</div>

Given `a.shape = (100, 1, 6, 3)` and `b.shape = (100, 1, 3)`, what is the shape of `a + b`?

**Step 1 - right-align:**
```
a: 100,   1,  6,  3
b: 100,   1,      3
```

**Step 2 - left-pad the shorter shape with 1s:**
```
a: 100,   1,  6,  3
b:   1, 100,  1,  3
```

**Step 3 - stretch any dimension that is 1:**
```
a: 100, 100,  6,  3
b: 100, 100,  6,  3
```

**Result: shape `(100, 100, 6, 3)` ✓**

> NOTE: The dimension of size 1 in `b` (originally position 0) gets stretched to match `a`'s 100. The dimension that was genuinely 100 in `b` (originally position 1) gets stretched in `a`'s dimension of size 1. This is why the answer is NOT `(100, 1, 6, 3)` - b's 100 fills a's 1.

<div class="alert alert-block alert-danger">

##### **4.7 NumPy pitfalls (week 4 lecture section)**

</div>

The lecture explicitly walks through 5 pitfalls:

**Pitfall 1 - Python loops over NumPy arrays.** The worst pattern:
```python
for i in range(len(x)):
    y[i] = x[i] * 2   # Slow: Python loop + per-element overhead
y = x * 2             # Fast: single vectorized C operation
```

**Pitfall 2 - calling NumPy functions inside Python loops.** Each call to `np.sin()`, `np.cos()` etc. has Python function-call overhead. If called $N^2$ times in a double loop, that overhead dominates:
```python
# Slow: N^2 calls to np.sin, each on a scalar
for i in range(N):
    for j in range(N):
        result[i,j] = np.sin(x[i]) * np.cos(y[j])

# Fast: 2 calls total, each operating on the whole array
result = np.sin(x[:, None]) * np.cos(y[None, :])
```

**Pitfall 3 - unintended copies.** As above, fancy indexing, Fortran reshapes, and explicit `.copy()` all allocate new memory.

**Pitfall 4 - intermediate arrays.** An expression like `a * b + c * d` creates two temporary arrays (`a*b` and `c*d`) before adding. For very large arrays this doubles memory usage and adds two extra passes through RAM. Libraries like NumExpr chunk the computation to fit into L1 cache to avoid this.

**Pitfall 5 - multi-threading with NumPy.** NumPy's C backend (BLAS/LAPACK) is already multithreaded internally for operations like `np.matmul`. If you also use Python `multiprocessing` on top, we can oversubscribe the CPU. Control via environment variables: `OMP_NUM_THREADS`, `MKL_NUM_THREADS`, `OPENBLAS_NUM_THREADS`.


<div class="alert alert-block alert-danger">

##### **4.8 The profiling → optimization loop (Haversine example)**

</div>

The exercise applies the two-profiler workflow to a geographic distance calculation. The pattern is a template for any optimization task:

1. Profile with `cProfile` → find that the distance function dominates
2. Profile with `kernprof` → find that line 27 (`np.sin`) takes $52\%$ of time
3. Fix: move the outer loop into a broadcasting operation → **$80\times$ speedup**
4. Re-profile → line 28 (`np.cos(p2)`) now dominates
5. Fix: pre-compute `np.cos(p2[:, 0])` outside the loop (it never changes inside) → another speedup
6. Fix: replace `arctan2(sqrt(a), sqrt(1-a))` with `arcsin(sqrt(a))` (mathematically identical, one fewer sqrt) → $\sim 25 \%$ further saving

> KEY INSIGHT: The number of NumPy *calls* matters as much as the size of each call. Reducing from $N^2$ calls to $N$ calls (each operating on $N$ elements) gives $O(N)$ speedup even if the total FLOPs are unchanged, because Python overhead per call is eliminated.



<div class="alert alert-block alert-danger">

##### **4.9 Strides cheat sheet**

</div>





```python
a = np.array([[1,2,3],[4,5,6]])  # shape (2,3), strides (12,4), int32

a.T            # shape (3,2), strides (4,12)   - VIEW
a.reshape(6)   # shape (6,),  strides (4,)     - VIEW (C-order)
a[::-1]        # shape (2,3), strides (-12,4)  - VIEW
a[::2]         # shape (1,3), strides (24,4)   - VIEW
a[[0,1]]       # shape (2,3)                   - COPY (fancy index)

np.shares_memory(a, b)   # True if same memory block
a.base is None           # True if a owns its data (not a view)
```

<div class="alert alert-block alert-danger">

##### **4.10 Python profiler**

(book notes)

</div>

You can run a python profiler with:
```bash
python -m cProfile -s comulative load.py [numbers idk]
```
With `-m`flag we execute the module (so we are running the `cProfile` module). Running this will give us profile statistis ordered by cumulative time.

We will have two columns called `percall`, and the first one states the time spent on the function excluding time spent on all the subcalls. The second one includes time spent on subcalls.

If we run something like:
```bash
python -m cProfile -o distance_chache.prof distance_script.py
```
Then the `-o`parameter specifies the file where the profiling information will be stored. Then to use SnakeViz to visualize we just to:
```bash
snakeviz distance_chache.prof
```

<div class="alert alert-block alert-danger">

##### **4.11 Using local caches**

(book notes)

</div>

Using local caches reduces network latency, for example, if we build local cache of the date, and while the first run of the code will take the same time, but the second will not require any network access.

For instance, the code in the book goes from 2.8 seconds to 0.26 seconds.

Cache management can be problematic and is common source of bugs. In the book example, files never change over time, but there are many use cases for caches where the source might be changing.

<div class="alert alert-block alert-danger">

##### **4.12 Line profiling**

(book notes)

</div>

We can use line profiling to understand the cost of each line of a code, so like:
```python   
@profile
def get_distance(p1, p2):
```
Then use `kernprof`from the `line_profiler` package so we run:
```bash
kernprof -l lprofile_distance_cache.py
```
Then we can look at results with:
```bash
python -m line_profiler lprofile_distance_cache.py.lprof
```

<div class="alert alert-block alert-danger">

##### **4.13 Profiling**

(book notes)

</div>

Pythons profiling only provides cumulative values per function as well as showing how much time is spent on subcalls.

In specific cases, it is possible to know if a subcall belongs to another function, but that is not possible in general.

Strategy used in the book is first, try the built-in Python profiling module `cProfile, since it is fast and provides high-level information.

If that is not enough, use line profiling = more informative but also slover.

<div class="alert alert-block alert-danger">

##### **4.14 FLOP counting from line profiler output (exam skill)**

</div>


The line profiler shows `Time` per line in $\mu s$. To calculate FLOP/s from profiler output:

**Step 1 - count FLOPs per iteration per line:**

| Expression | FLOPs | Reason |
|---|---|---|
| `x[i] * x[i]` | 1 | one multiply |
| `x[i] * x[i] + 4` | 2 | multiply + add |
| `y[n-i-1] / x[i]` | 1 | one divide |
| `z + a / b` | 2 | divide + add |
| `np.sqrt(x)` on N elements | N | one sqrt per element |
| `a[i]` indexing | 0 | not arithmetic |
| `z = ...` assignment | 0 | not arithmetic |

**Step 2 - multiply by Hits (from profiler)**

**Step 3 - sum total time from the Time column, convert µs → seconds** (`÷ 1,000,000`)

**Step 4 - FLOP/s = total FLOPs / total seconds**


<div class="alert alert-block alert-danger">

##### **4.15 kernprof output: runtime scaling to larger workloads (exam skill)**

</div>


The 2025 exam Q11 gives a profiler output for a *small* dataset and asks the runtime
for the *normal* (larger) workload. The key: **not every line scales with data size**.

**Procedure:**

**Step 1 - identify which lines scale and which are fixed overhead.**

> Lines inside the loop body (called `n_items` times) scale with dataset size.

> Lines called once (e.g. `prep_conds(params)`, `result = []`) are fixed overhead.

**Step 2 - read per-item cost from profiler.**

For each variable line: `cost_per_item = Time_µs / Hits`

**Step 3 - scale and add back fixed overhead.**

`T_normal = T_fixed + sum(cost_per_item × n_normal)` (convert µs → s by ÷ 1 000 000)

**Worked example (2025 exam Q11):**

```
Line  Hits    Time(µs)   Per Hit   Contents
 99      1  2005064.0  2.0e+06   conds = prep_conds(params)  ← fixed
100      1        4.0      4.0   result = []                 ← fixed
101   1001      740.0      0.7   for x in data:
102   1000  1266748.0   1266.7   y = process_single(x, conds)  ← scales
103   1000     1685.0      1.7   result.append(y)              ← scales
```

Fixed: $(2005064 + 4) \times 10^{-6} = 2.005$ s

Variable per item: $(1266748 + 1685) / 1000 \times 10^{-6} = 0.001268$ s/item

Normal (10 000 items): $2.005 + 0.001268 \times 10000 \approx \mathbf{14.7}$ s ✓

> KEY: The `Hits` column tells you the profiling dataset size. Divide variable-line `Time` by `Hits` to get per-item cost, multiply by target size, add fixed lines back.


<div class="alert alert-block alert-danger">

##### **4.16 "More vectorised" $\neq$ always faster**

</div>


The fully vectorised no-loop Haversine creates intermediate arrays of shape `(N, N, 2)`.
For $N=5000$, that is $5000 \times 5000 \times 2 \times 8$ bytes = **400 MB of RAM** that must be written and read.

The one-loop version holds only one row at a time in memory -  it stays in cache.

| N | Faster version | Why |
|---|---|---|
| Small (fits in L2) | No-loop | No Python overhead at all |
| Large ($> \sim 350\times350$) | One-loop | Intermediate arrays trash RAM |

**KEY EXAM INSIGHT:** We cannot predict which is faster from reading the code. We must **measure**. This is the same lesson as Week 3: performance depends on data size and hardware, not just algorithm structure.


<div class="alert alert-block alert-danger">

##### **4.17 Specifying CPU model for reproducible timing**

</div>

On a shared HPC cluster, different nodes have different CPUs and therefore different performance characteristics. To get reproducible profiling results, pin to a specific CPU model in your job script:

```bash
#BSUB -R "select[model==XeonGold6226R]"
```

Without this, the same script submitted twice may run on different hardware and give different timings, making comparison meaningless.
```

<div class="alert alert-block alert-info">

### **Week 5**

Parallelism Part 1
</div>


<div class="alert alert-block alert-danger">

##### **5.1 Why parallelism is needed**

</div>

Modern CPUs have mostly stopped getting faster clock-for-clock. Instead, manufacturers add more cores. To benefit from this, programs must run multiple tasks simultaneously. The central question of this week is: how much faster can we actually go?


<div class="alert alert-block alert-danger">

##### **5.2 Concurrency vs parallelism**

</div>

These two terms are often confused:

> **Concurrency** is when tasks *overlap in time* so they may not run at the exact same instant, but they make progress together (e.g. one task waits for I/O while another runs).

> **Parallelism** is when tasks *execute at the exact same instant* on multiple cores or processors.

Parallelism is a subset of concurrency. For HPC, we care primarily about parallelism.


<div class="alert alert-block alert-danger">

##### **5.3 The Global Interpreter Lock (GIL)**

</div>

CPython (the standard Python interpreter) has a lock called the GIL that allows only **one thread to execute Python bytecode at a time**, even on a multicore machine.

```python
import threading

# This looks parallel but is NOT -  threads take turns, serialised by the GIL
t1 = threading.Thread(target=heavy_python_function)
t2 = threading.Thread(target=heavy_python_function)
t1.start(); t2.start()
# Result: same speed as running sequentially
```

> PROS of GIL: prevents race conditions automatically, no need for manual locks in single-threaded code.

> CONS of GIL: CPU-bound multithreaded Python code cannot run in true parallel.

**When threading CAN work**, some operations manually release the GIL:

> File I/O and network operations

> NumPy C-level operations (e.g. `np.matmul`, `np.sin` on large arrays)

> Other C extensions that explicitly release it

So `ThreadPool` is valid for parallelising many `np.matmul` calls because NumPy releases the GIL during BLAS (where BLAS means Basic Linear Algebra Subprograms) computation, letting threads run truly simultaneously.

**The general solution: `multiprocessing`**, so instead of threads sharing one interpreter, spawn entirely separate Python processes, each with its own GIL and memory space. No GIL conflict.

Trade-off is more overhead (process startup, data must be pickled/copied to pass between processes, no shared memory by default).

> NOTE: PEP 703 (accepted, Python 3.13+) is making the GIL optional. For this course, assume the GIL is present.



<div class="alert alert-block alert-danger">

##### **5.4 Embarrassingly parallel problems**

</div>


A problem is **embarrassingly parallel** if it can be split into completely independent subtasks with no communication needed between them. These are the easiest to parallelise and give the best speedups.

Examples: Monte Carlo simulations, image processing where each pixel is independent, applying the same function to each row of a dataset.

Problems that require **communication** (e.g. particles that interact with their neighbours) are harder, communication cost can dominate and kill the speedup.

<div class="alert alert-block alert-danger">

##### **5.5 `multiprocessing.Pool`, the main tool**

</div>



```python
import multiprocessing as mp

def f(x):
    return x * x

with mp.Pool(processes=4) as pool:
    results = pool.map(f, [1, 2, 3, 4, 5, 6, 7, 8])
```

`Pool(n)` creates `n` worker processes. `pool.map(f, data)` distributes items in `data` across the workers, collects results, and blocks until all are done. It is the idiomatic choice for embarrassingly parallel work.

```python
# apply_async -  lower level, non-blocking submission
results_async = [pool.apply_async(f, (x,)) for x in data]
results = [r.get() for r in results_async]  # blocks here

# pool.map with chunksize -  batches submissions automatically
results = pool.map(f, data, chunksize=1000)

# starmap -  like map but unpacks argument tuples
results = pool.starmap(f, [(a1, b1), (a2, b2), ...])
```

> KEY DIFFERENCE: `pool.map` is **eager** (blocks until everything is done). Python's built-in `map` is **lazy** (returns immediately, computes on demand). They are not equivalent.


<div class="alert alert-block alert-danger">

##### **5.6 Parallelisation overhead, the hidden cost**

</div>

Spawning processes, pickling data, and sending it over inter-process pipes all take time. If each task is tiny, this overhead **dominates** and parallelism makes things *slower*:

```python
# BAD: 1,000,000 separate apply_async calls, each doing 1 sample
# Overhead per call >> work per call → slower than serial
results = [pool.apply_async(sample_one, ()) for _ in range(1_000_000)]

# GOOD: 4 calls, each doing 250,000 samples -  overhead negligible
results = [pool.apply_async(sample_many, (250_000,)) for _ in range(4)]
```

**Solution: chunking.** Give each worker a large enough chunk of work that the overhead is negligible compared to the computation.

We can diagnose this using the `time` command:
```bash
time python pi.py
# real    0m5.2s   ← wall-clock time
# user    0m18.4s  ← total CPU time across all cores
```
If `user >> real`: multiple cores are genuinely running in parallel (good). If `real ≈ user` despite using multiple processes: overhead has serialised execution (bad).

<div class="alert alert-block alert-danger">

##### **5.7 Load balancing**

</div>

Even with chunking, we can still get a poor speedup if work is **unevenly distributed**:

**Static scheduling** is to divide all work upfront into `n_proc` equal chunks and assign one to each worker. Fast to set up, but if some chunks take longer than others, fast workers sit idle waiting for the slow one. Overall speed = speed of the slowest chunk.

**Dynamic scheduling** is to create many more chunks than workers. When a worker finishes its chunk, it immediately picks up the next one from a queue. Automatically balances the load.

```python
# Static (bad for uneven work like Mandelbrot):
chunk_size = len(points) // n_proc  # n_proc chunks total
chunks = [points[i*chunk_size:(i+1)*chunk_size] for i in range(n_proc)]

# Dynamic (good): many small chunks, pool self-schedules
pool.map(f, points, chunksize=100)  # many chunks, workers grab as needed
```

The Mandelbrot set is the canonical example of uneven work: points inside the set require all 100 iterations to classify, points far outside escape in 1–2 iterations. Equal-sized chunks give wildly different runtimes per worker.

> KEY INSIGHT: Reducing the effective serial bottleneck (idle time from load imbalance) is often more impactful than adding more processors. A measured parallel fraction of 0.8 (static chunks) vs 0.98 (small chunks) changes the Amdahl ceiling from $5 \times$ to $50 \times$.


<div class="alert alert-block alert-danger">

##### **5.8 Amdahl's law (recap in the parallelism context)**

</div>

$$S(p) = \frac{1}{B + \frac{1-B}{p}} = \frac{1}{(1-F) + \frac{F}{p}}$$

The four reasons a parallel program is slower than expected, as stated directly in the lecture:

> **1. Parallelisation overhead**, process startup, pickling, IPC. Fix: chunking.

> **2. Too much communication**, processes needing to share intermediate results. Fix: restructure algorithm to minimise communication.

> **3. Load imbalance**, some workers finish early and sit idle. Fix: dynamic scheduling with small chunks.

> **4. Amdahl's law**, the serial fraction $B$ sets an absolute ceiling regardless of cores. Fix: reduce the serial fraction (harder, requires algorithmic change).

<div class="alert alert-block alert-danger">

##### **5.9 Parallelism tool decision flowchart**

</div>

The exam gives a scenario and asks which approach improves performance most. Work through
this in order:

**Step 1 - is the bottleneck I/O or CPU?**

> I/O bound (waiting on disk, network): threading or asyncio. The GIL does not matter
  because threads release it while waiting on I/O. Multiprocessing adds unnecessary
  process-spawn overhead.

> CPU bound: go to step 2.

**Step 2  - is the CPU-bound code Python or NumPy/C?**

> Pure Python loops: the GIL blocks real thread parallelism. Use **multiprocessing**
  (`Pool.map`). Threads will not help.

> NumPy / BLAS operations (matmul, dot, SVD): NumPy releases the GIL during C-level
  computation. Use **BLAS threads** (`OMP_NUM_THREADS`) or **ThreadPool**  - both work
  because the GIL is not held during the heavy work.

**Step 3  - are iterations independent?**

> Sequential dependency (each step needs the result of the previous): cannot parallelise
  the inner loop. Parallelise across independent calls instead (different inputs, different
  simulations).

> Independent iterations: parallelise freely.

**Step 4  - are task durations uniform or variable?**

> Uniform: static scheduling (large chunksize) is fine, low overhead.

> Variable (some tasks take much longer): **dynamic scheduling** (`chunksize=1` or small
  chunksize). Static scheduling leaves fast workers idle while slow workers finish.

**Step 5  - is the problem trivially parallel with no data dependencies per element?**

> Yes, and the array is large: consider **GPU**. Massive throughput for embarrassingly
  parallel element-wise work. Only worth it if transfer overhead is amortised over many
  iterations.

| Situation | Right tool |
|---|---|
| I/O bound | `threading` or `asyncio` |
| CPU-bound Python loops | `multiprocessing.Pool` |
| CPU-bound NumPy (matmul etc.) | BLAS env vars or `ThreadPool` |
| Sequential dependencies | Parallelise *across* independent calls |
| Variable task times | `Pool.map(chunksize=1)` (dynamic) |
| Embarrassingly parallel, large data, many iterations | GPU (Numba `@cuda.jit`) |

EXAM TRAP: threading with Python code and the GIL → no speedup for CPU-bound work. Threading with NumPy → real speedup because BLAS releases the GIL.

<div class="alert alert-block alert-danger">

##### **5.10 Fitting Amdahl's law to measured data**

</div>


When we have a measured speedup plot from the exercise, you can back out the parallel fraction:

$$F = \frac{p\left(1 - \frac{1}{S(p)}\right)}{p - 1}$$

For example, if we measure $S(8) = 6.5$ with $p = 8$:
$$F = \frac{8(1 - 1/6.5)}{8-1} = \frac{8 \times 0.846}{7} \approx 0.967$$

Then the theoretical maximum: $S(\infty) = 1/(1-0.967) \approx 30$.

Real speedup plateaus below this because Amdahl's model ignores process startup overhead, memory bandwidth limits, and OS scheduling noise.


<div class="alert alert-block alert-danger">

##### **5.11 The `time` command, reading real vs user**

</div>


```bash
time python script.py
# real    0m2.1s   ← total wall-clock time you waited
# user    0m7.8s   ← total CPU-seconds consumed across all cores
# sys     0m0.1s   ← time in OS kernel calls
```

> `user > real` → true parallelism: multiple cores ran simultaneously (user time accumulated faster than wall time)

> `user ≈ real` → little parallelism: effectively one core at a time despite multiple processes

> `user < real` → processes spent time waiting (I/O bound, or too much IPC overhead)


<div class="alert alert-block alert-danger">

##### **5.12 Chunksize trade-off summary**

</div>



| Chunk size | Effect |
|---|---|
| Too small (e.g. 1) | High overhead: many IPC round-trips, overhead > work |
| Too large (e.g. all data in one chunk) | No load balancing: one process does everything |
| Sweet spot | Workers stay busy, overhead is $<5\%$ of total runtime |

Rule of thumb: aim for at least $10–100 \times$ more chunks than workers. Let the pool scheduler handle the rest.



<div class="alert alert-block alert-danger">

##### **5.13 Useful commands for Week 5 exercises**

</div>




```python
import multiprocessing as mp
import os

mp.cpu_count()                # number of logical CPUs (may include hyperthreads)
len(os.sched_getaffinity(0))  # cores actually available to this process (more reliable)

with mp.Pool(n) as pool:
    results = pool.map(f, data, chunksize=k)      # simple map
    results = pool.starmap(f, list_of_tuples)     # unpack args
    r = pool.apply_async(f, (arg,)); r.get()      # non-blocking
```

<div class="alert alert-block alert-danger">


##### **5.14 Parallelism formulas summary**

</div>

Ahmdal's law:
$$
S(p) = \frac{1}{(1-F)+\frac{F}{p}} 
$$
$$
= \frac{1}{B+\frac{1-B}{p}}
$$
With $B=1-F$ and $F$ is the parallel fraction while $B$ is the serial fraction, and $p$ is the number of processors.

Here is the theoretical maximum speedup as $p \to \infty$:
$$
S(\infty) = \frac{1}{B} = \frac{1}{1-F}
$$

If we want to know the parallel fraction $F$ given the speedup $S$ and number of processors $p$, we can rearrange Amdahl's law to get:
$$
F = \frac{p(1-\frac{1}{S(p)})}{p-1}
$$
WHEN AND HOW CAN WE KNOW if it is worth parallelize the program? For instance, if someone wants a speedup of at least 4 using 8 processors, and $F=0.8$ meaning $80 \%$ of the program can be parallelized, then we can calculate the speedup using Amdahl's law:
$$
S(8) = \frac{1}{(1-0.8)+\frac{0.8}{8}} = 3.57
$$
So in this case, it would not be worth to parallelize the program because we would not achieve the desired speedup of 4. 

<img src="SymPyBilleder/2026-05-24-13-19-04.png" width="550">

<div class="alert alert-block alert-danger">

##### **5.15 Amdahl's law ALL forms**

</div>


**Standard form**, given parallel fraction $F$:
$$
S(p) = \frac{1}{(1-F)+\frac{F}{p}}
$$
Where $F$ = parallel fraction, $p$ = number of processors.


**Serial fraction form**, substituting $B = 1 - F$:
$$
S(p) = \frac{1}{B+\frac{1-B}{p}}
$$


**Theoretical maximum**, as $p \to \infty$:
$$
S(\infty) = \frac{1}{B} = \frac{1}{1-F}
$$
Even with infinite processors, speedup is bounded by the serial fraction.


**Inverse form**, solve for $F$ given measured $S(p)$ and $p$:
$$
F = \frac{p\!\left(1 - \frac{1}{S(p)}\right)}{p - 1}
$$


**Solve for $B$**, directly from the serial-fraction form:
$$
B = \frac{\frac{1}{S(p)} - \frac{1}{p}}{1 - \frac{1}{p}}
$$

**Efficiency**, fraction of processors doing useful parallel work:
$$
E(p) = \frac{S(p)}{p} = \frac{1}{p(1-F)+F}
$$
$E \in (0, 1]$; decreases as $p$ grows, reflecting diminishing returns.

> **Note:** Amdahl's law assumes a fixed problem size and ignores parallelisation overhead and memory bandwidth limits. Real speedup is always $\leq$ the theoretical value.

<div class="alert alert-block alert-danger">

##### **5.16 Amdahl's law interactive solver (exam tool)**

</div>

In [ ]:
def amdahl_solve(p=None, S_p=None, F=None):
    """
    Solve Amdahl's law for any unknown given the other two.
    Always prints S_max and, if p and F known, S(p).

    Examples
    --------
    amdahl_solve(p=3, S_p=2.5)   # back out F, then S_max  (Q8 2025 exam)
    amdahl_solve(p=8, F=0.9)     # predict speedup on 8 cores
    amdahl_solve(F=0.95)         # just show S_max
    """
    if F is None and p is not None and S_p is not None:
        F = p * (1 - 1/S_p) / (p - 1)
    if F is not None and 0 < F < 1:
        S_max = 1 / (1 - F)
    else:
        S_max = None

    print("=" * 42)
    if p    is not None: print(f"  p (processors)   = {p}")
    if F    is not None: print(f"  F (parallel frac)= {F:.6f}")
    if S_p  is not None: print(f"  S({p}) measured  = {S_p}")
    if S_max is not None: print(f"  ➜ S(∞) max       = {S_max:.3f}×")
    if p is not None and F is not None:
        s = 1 / ((1 - F) + F/p)
        print(f"  ➜ S({p}) predicted = {s:.3f}×")
        print(f"  ➜ Efficiency E({p}) = {s/p:.3f}  ({s/p*100:.1f}%)")
    print("=" * 42)

# 2025 exam Q8: p=3, S(3)=2.5 → find F and S_max
amdahl_solve(p=3, S_p=2.5)
print()
# What speedup with F=0.9 on 16 cores?
amdahl_solve(p=16, F=0.9)

  p (processors)   = 3
  F (parallel frac)= 0.900000
  S(3) measured  = 2.5
  ➜ S(∞) max       = 10.000×
  ➜ S(3) predicted = 2.500×
  ➜ Efficiency E(3) = 0.833  (83.3%)

  p (processors)   = 16
  F (parallel frac)= 0.900000
  ➜ S(∞) max       = 10.000×
  ➜ S(16) predicted = 6.400×
  ➜ Efficiency E(16) = 0.400  (40.0%)


<div class="alert alert-block alert-danger">

##### **5.17 Working directly with T_serial and T_parallel (CRITICAL exam skill)**

</div>


Amdahl's formula is derived from a simple decomposition of runtime. This direct form is more useful for many exam questions than the formula itself:

$$T(p) = T_{\text{serial}} + \frac{T_{\text{parallel}}}{p}$$

$$S(p) = \frac{T(1)}{T(p)} = \frac{T_{\text{serial}} + T_{\text{parallel}}}{T_{\text{serial}} + T_{\text{parallel}}/p}$$

The parallel fraction is just:
$$F = \frac{T_{\text{parallel}}}{T_{\text{serial}} + T_{\text{parallel}}}$$




The parallel fraction is just:
**Worked example (2024R exam Q3 and Q4):**

A program takes 10 minutes on 4 processors, with $F = 0.8$.

*Q3: Time on 1 processor?*

$$S(4) = \frac{1}{0.2 + 0.8/4} = 2.5 \quad \Rightarrow \quad T(1) = 10 \times 2.5 = 25 \text{ min}$$

*Q4: Reduce the serial part by 3 minutes. New runtime on 4 processors?*

The key insight is: **you do not need to recalculate F or use the formula**. The runtime on p processors is $T_{\text{serial}} + T_{\text{parallel}}/p$. Reducing $T_{\text{serial}}$ by 3 simply gives:

$$T_{\text{new}}(4) = (T_{\text{serial}} - 3) + \frac{T_{\text{parallel}}}{4} = 10 - 3 = 7 \text{ min}$$

This works because the original equation was $T_{\text{serial}} + T_{\text{parallel}}/4 = 10$, so replacing $T_{\text{serial}}$ with $T_{\text{serial}} - 3$ gives $10 - 3 = 7$.

> KEY EXAM RULE: When only one component of the runtime changes (serial OR parallel, not both), you can directly add/subtract from the current runtime on p processors without touching the Amdahl formula at all.

<img src="SymPyBilleder/2026-05-21-21-57-43.png" width="550">


<div class="alert alert-block alert-danger">

##### **5.18 Comparing strategies: more processors vs less serial work**

</div>


From the Week 5 exercise ($20 \:s$ serial + $100 \:s$  parallel, currently 4 processors):

| Strategy | Calculation | Runtime |
|---|---|---|
| Reduce T_serial: $20 \:s \rightarrow 5 \:s$  , keep $p=4$ | $5 + 100/4$ | **$30 \: s$** |
| More processors:  $4 \rightarrow 8$  , keep T_serial $=20 \:s$ | $20 + 100/8$ | $32.5 \: s$ |

Reducing the serial bottleneck wins here. This is Amdahl's core message: **the serial fraction is the ceiling, not the number of processors**.

> NOTE: This comparison pattern - "which gives better runtime: optimise the serial part or add more cores?" - appears directly in the Week 5 exercise and is a common exam question type.


<div class="alert alert-block alert-danger">

##### **5.19 GIL exception: when threading IS appropriate**

</div>

The GIL blocks parallel execution of Python bytecode. But it does NOT block operations that release the GIL at the C level. This is the critical exception:

> **NumPy BLAS operations** (`np.matmul`, `np.dot`, SVD, etc.) release the GIL during computation. `ThreadPool` genuinely runs multiple `np.matmul` calls in parallel.

> **I/O operations** (file reads, network calls) also release the GIL.

So the rule is:

| Task | Use threading? | Why |
|---|---|---|
| Pure Python loops | ✗ | GIL serialises all threads |
| `np.matmul` on large arrays | ✓ | BLAS releases the GIL |
| Reading files concurrently | ✓ | I/O releases the GIL |
| Custom Python computation | ✗ | Use multiprocessing instead |

EXAM PATTERN (Q17 in 2024R): "Is threading appropriate for summing rows with `np.sum()`?" - **YES**, because `np.sum` is a C-level NumPy operation that releases the GIL. Threads genuinely run in parallel.


<div class="alert alert-block alert-danger">

##### **5.20 Parallel reduction time complexity (exam calculation type)**

</div>




This appears in the context of reduction operators (Week 6) but the time analysis is a Week 5/6 exam question type.

**Sequential parallel approach** (e.g. sum each row in parallel, then sum the row sums serially):

> Parallel part: $n$ row sums, all done simultaneously → time proportional to $n$ (one row sum)

> Serial part: sum $n$ row sums → time proportional to $n$

> Total time: $n + n = 2n$

**Binary tree reduction** over $N = n^2$ elements:

> Takes $\log_2(N) = \log_2(n^2) = 2\log_2(n)$ rounds

> Each round is fully parallel

Speedup of tree reduction vs sequential parallel approach:

$$\text{Speedup} = \frac{2n}{2\log_2(n)} = \frac{n}{\log_2(n)}$$

> EXAMPLE: For $n = 1000$, speedup $= 1000/\log_2(1000) \approx 1000/10 = 100\times$.

> NOTE: This assumes unlimited processors. The tree reduction becomes even more attractive as $n$ grows because $n/\log_2(n) \to \infty$.

<div class="alert alert-block alert-info">

### **Week 6**

Parallelism Part 2
</div>


<div class="alert alert-block alert-danger">

##### **6.1 Reduction operators**

</div>

A **reduction operator** takes an array of elements and collapses them 
into a single result (e.g. sum, product, logical AND). Not every operator 
qualifies. To be usable in a parallel framework, an operator $\circ$ must 
satisfy **both** of the following properties:

> **1. Commutative:** $a \circ b = b \circ a$, order of operands does not 
matter (swap the inputs - same result?)

> **2. Associative:** $(a \circ b) \circ c = a \circ (b \circ c)$, grouping 
of operands does not matter (move the brackets - same result?)

**Why both properties are needed:**

> **Associativity** is needed to *split the work* into subtasks, it lets us 
regroup the computation into independent pairs without changing the result.

> **Commutativity** is needed by many frameworks that may *reorder* inputs 
for implementation efficiency. The simple binary tree algorithm does not 
actually reorder operands, so strictly only associativity is required for 
it, but most frameworks require both, so treat commutativity as a 
requirement in exam answers.

SO IN SHORT, both are needed because:

> The tree redistributes work across processes in arbitrary order (needs commutativity)

> Different processes reduce different sub-groups first (needs associativity)


**Examples of valid reduction operators:**

| Operation | Commutative? | Associative? | Usable in reduction? |
|---|---|---|---|
| Addition (+) | ✓ | ✓ | ✓ YES |
| Multiplication (×) | ✓ | ✓ | ✓ YES |
| Maximum / Minimum | ✓ | ✓ | ✓ YES |
| Set union (∪) | ✓ | ✓ | ✓ YES |
| Set intersection (∩) | ✓ | ✓ | ✓ YES |
| Subtraction (−) | ✗ | ✗ | ✗ NO |
| Division (÷) | ✗ | ✗ | ✗ NO |
| Matrix multiplication | ✗ | ✓ | ✗ NO (not commutative) |
| np.mean (average) | ✓ | ✗ | ✗ NO* |

*`np.mean` is NOT associative: mean(mean([1,2]), mean([3])) ≠ mean([1,2,3]). Use sum reduction + divide at the end instead.


> KEY NOTE: The power function is the canonical non-example. 
`(2^1)^2 = 4` but `2^(1^2) = 2` - changing grouping changes the result, 
so not associative.

> KEY NOTE: `abs(x+y)` is the exam trap (2024 Q6). It looks like addition 
but the `abs` wrapping breaks associativity. Counterexample with 
`1, 2, -3`: `||1+2|+(-3)| = 0` but `|1+|2+(-3)|| = 2` - different 
grouping, different result. **Fix:** do a normal parallel sum reduction 
first, apply `abs` once at the very end.

> GENERAL RULE: any operator that wraps a valid reduction with a non-linear 
function (`abs`, `sqrt`, `log`, etc.) likely breaks associativity. Always 
test with a counterexample before assuming it works.

> KEY EXAM RULE: check both properties. A function that fails either one cannot be used directly in the tree reduction framework. The fix for non-associative operations is usually to restructure: reduce a different quantity (e.g. sum instead of mean) and post-process.

<div class="alert alert-block alert-danger">

##### **6.2 The binary tree reduction algorithm**

</div>

Instead of summing $N$ elements sequentially ($N−1$ serial additions), the tree reduction halves the active range each round, doing all additions in parallel within each round:

```
N = 8 elements: [a, b, c, d, e, f, g, h]

Round 1 (4 parallel additions):
  a += b,  c += d,  e += f,  g += h
  → [a+b, c+d, e+f, g+h]

Round 2 (2 parallel additions):
  a += c,  e += g
  → [a+b+c+d, e+f+g+h]

Round 3 (1 addition):
  a += e
  → [a+b+c+d+e+f+g+h]
```

**Complexity:** $\log_2(N)$ rounds instead of $N-1$ steps. For $N = 100{,}000$ images: 17 rounds instead of 99,999 sequential additions. Each round is fully parallelisable.

The template code expresses one step using a strided slice:

```python
def reduce_step(args):
    b, e, s, elemshape = args
    arr = tonumpyarray(shared_arr).reshape((-1,) + elemshape)
    arr[b:e:s] += arr[b + s//2 : e : s]
```

`b`, `e`, `s` are the start, end, and stride for this round. The full reduction calls `pool.map` once per round with a halving stride. Each call to `pool.map` is the synchronisation barrier between rounds, workers cannot start round 2 until round 1 is complete, because round 2's inputs are round 1's outputs.

<div class="alert alert-block alert-danger">

##### **6.3 Why shared memory is essential here**

</div>

Normal `multiprocessing` sends data between processes by pickling and copying it over an inter-process pipe. For 100,000 images of shape $218 \times 178 \times 3$ at float32, the total data is $\sim 10 \: GB$. Copying this on every round would be catastrophically slow and defeat the purpose of parallelism.

`mp.RawArray` allocates a block of memory that **all worker processes can access directly** so no copying, no pickling:

```python
import multiprocessing as mp
import ctypes
import numpy as np

# Allocate shared memory
shared_arr = mp.RawArray(ctypes.c_float, data.size)

# Wrap it as a NumPy array (zero-copy view)
arr = np.frombuffer(shared_arr, dtype=np.float32).reshape(data.shape)
np.copyto(arr, data)  # copy data in once

# Worker processes access the same physical memory via a global reference
# set by the pool initialiser function
```

Workers write results directly into the shared array. No inter-process data transfer happens during computation, just `pool.map` calls to coordinate which indices each worker handles in each round.

> KEY INSIGHT: Shared memory trades the safety of isolated processes for performance. Since all workers write to different indices in each round (no two workers touch the same element), there are no race conditions even without locks.

<div class="alert alert-block alert-danger">

##### **6.4 Non-Uniform Memory Access (NUMA)**

</div>

This is the key new concept of Week 6. When running the tree reduction, the speedup plot shows a surprising result: performance improves as you add cores up to $\sim 24$, then **drops** when you go beyond that.

**Why:** A server-grade HPC node (e.g. Xeon Gold 6342) is physically **two separate CPU chips (sockets)** on one motherboard. Each socket has its own local RAM bank. The first 24 cores sit on socket 0, the next 24 on socket 1.

```
Socket 0 (cores 00–23) ←→ RAM bank 0
Socket 1 (cores 24–47) ←→ RAM bank 1
           ↑                        
inter-socket interconnect (slower)  
```

When only socket 0's cores run, all memory reads/writes go to RAM bank 0, fast local access. When socket 1's cores also run, they need to access the shared array which was allocated in socket 0's RAM bank. Every memory access by socket 1 must travel over the slower inter-socket link. This **cross-NUMA memory access** causes the performance drop beyond 24 cores.

> NOTE: This is called Non-Uniform Memory Access because the time to access RAM is not uniform -  it depends on which socket the memory is physically allocated in, relative to which core is accessing it.


**The 3 NUMA traps the exam likes:**

**Trap 1 - "adding more cores always helps"**
→ False. Beyond the socket boundary (~24 cores), performance can *drop* due to cross-NUMA memory access.

**Trap 2 - "numactl fixes single-core performance"**
→ False. `--interleave=all` only helps *scaling* across sockets. It doesn't speed up single-core or small-core-count runs - average latency stays similar or slightly increases.

**Trap 3 - "the fix is to use fewer cores"**
→ Not the right fix. The fix is `numactl --interleave=all` which distributes memory pages round-robin across both RAM banks so neither socket has all-remote accesses.


**One-line summary to memorise:**

> Performance drops beyond ~24 cores because memory was allocated on socket 0's RAM, so socket 1's cores pay inter-socket latency on every access. Fix: `numactl --interleave=all` balances pages across both banks.

Your notes already have all of this - I just wanted to make the exam trap framing explicit. You're good on this topic.

<div class="alert alert-block alert-danger">

##### **6.5 Fix: `numactl --interleave=all`**

</div>

```bash
numactl --interleave=all python reduction.py <path>
```

`--interleave=all` tells the OS to distribute memory pages in a **round-robin fashion across all NUMA nodes**. Instead of the entire 10 GB shared array sitting in socket 0's RAM, pages alternate between bank 0 and bank 1:

> Socket 0's cores: half of accesses are local (fast), half are remote (slow)

> Socket 1's cores: half of accesses are local (fast), half are remote (slow)

Both sockets now have symmetric, balanced access. No single socket is a bottleneck. The speedup curve continues growing beyond 24 cores.

> IMPORTANT: `--interleave=all` fixes **scaling** (the curve stops dropping) but does not improve single-core speed -  average latency stays similar or slightly increases for small datasets. The benefit only appears when using many cores across both sockets.



<div class="alert alert-block alert-danger">

##### **6.6 Simple (sequential) vs binary tree reduction - complexity comparison**

</div>


| Method | Steps | Parallelisable? |
|---|---|---|
| Sequential sum | $N - 1$ | No, each step depends on previous |
| Binary tree | $\log_2(N)$ | Yes, all additions within a round are independent |

For $N = 100{,}000$: sequential = 99,999 steps vs tree = 17 rounds. Each of the 17 rounds can use all available cores.


<div class="alert alert-block alert-danger">

##### **6.7 When is a loop parallelisable? (formal definition)**

</div>

A loop is parallelisable if and only if **no iteration depends on the output of another iteration**. Formally, iteration $i$ and iteration $j$ are independent if:

> The input to iteration $i$ is not affected by the output of iteration $j$, and vice versa.

**The one-question test:** could you run iteration 50 without having run iterations 1–49 first? If yes → parallelisable. If no → sequential dependency, cannot parallelise.

**Three patterns to recognise instantly:**

| Pattern | Example | Parallelisable? |
|---|---|---|
| Output fed back as next input | `x = f(x)` inside loop | ✗ No |
| Each iteration writes to its own output slot | `out[i] = f(data[i])` | ✓ Yes |
| Each iteration only reads shared data | `frame = render(scenes[i])` | ✓ Yes |


**Exam trap:** a variable defined *before* the loop is not the issue. The issue is whether that variable is **overwritten inside the loop and then read again in the next iteration**.

<div class="alert alert-block alert-danger">

##### **6.8 Loop order and spatial locality (exam Q21 trap)**

</div>


The 2025 exam Q21 asks which statement about a double-loop is **not** correct.
The trap is *"the order of the loops can be switched with no impact"* - that is **false**.

```python
for i in range(N):
    for j in range(N):
        forces[i, j] = G * m0 * m1 / (r * r)
```

`forces` is stored row-major (C order). With `i → j`:
- Inner loop writes `forces[i, 0], forces[i, 1], ...` - same row, **contiguous in memory** → cache-friendly.

With `j → i` (swapped):
- Inner loop writes `forces[0, j], forces[1, j], ...` - skipping N floats per step → **cache-unfriendly, many cache misses**.

**Summary of Q21 options:**

| Option | Statement | Correct? |
|---|---|---|
| A | `@jit` decorator improves performance | ✓ Yes - JIT compiles loops to machine code |
| B | Loops can be easily parallelized | ✓ Yes - no dependencies between `(i,j)` pairs |
| C | Can be improved using only NumPy | ✓ Yes - vectorized array ops replace the loop |
| D | Loop order can be switched with no impact | ✗ **No** - changes cache access pattern |

> Rule: loop order is never free. Innermost loop should walk memory in the fastest-varying dimension (last index for C-order arrays).



A loop is parallelisable if and only if **no iteration depends on the output of another
iteration**. Formally, iteration $i$ and iteration $j$ are independent if:

> The input to iteration $i$ is not affected by the output of iteration $j$, and vice versa.

**The one-question test:** could you run iteration 50 without having run iterations 1–49
first? If yes → parallelisable. If no → sequential dependency, cannot parallelise.

**Three patterns to recognise instantly:**

| Pattern | Example | Parallelisable? |
|---|---|---|
| Output fed back as next input | `x = f(x)` inside loop | ✗ No |
| Each iteration writes to its own output slot | `out[i] = f(data[i])` | ✓ Yes |
| Each iteration only reads shared data | `frame = render(scenes[i])` | ✓ Yes |

**Exam trap:** a variable defined *before* the loop is not the issue. The issue is whether
that variable is **overwritten inside the loop and then read again in the next iteration**.

```python

<div class="alert alert-block alert-danger">


##### **6.9 Why tree reduction does not dramatically beat `np.sum`**

</div>

A natural question from the exercise: does the parallel tree reduction beat `np.sum(data, axis=0)`? The answer is: faster, but not dramatically -  roughly 2–4 x at best. This is because:

> `np.sum` is already highly optimised C code exploiting SIMD (vector CPU instructions)

> Python multiprocessing has process coordination overhead for each of the $\log_2(N)$ rounds

> Writing parallel Python that substantially beats single-threaded optimised C is genuinely hard

This reinforces the course's central lesson: **always profile and measure first**. Raw parallelism is not a free performance multiplier.


<div class="alert alert-block alert-danger">

##### **6.10 Exam checklist for Week 6**

</div>

The 2025 exam (Q3) asks directly: "Can set intersection be used in a parallel reduction framework?" the answer is yes, because it is both commutative and associative. Practice checking any operator against both properties before answering.

The 2024 exam asks about a function that cannot be used in a tree reduction framework and asks what the engineer should do instead, the answer is always: check commutativity and associativity, then restructure the algorithm if needed.

```python
# Summary of useful commands
numactl --interleave=all python script.py   # fix NUMA scaling
time python script.py                       # measure real vs user time

# Shared memory pattern
shared_arr = mp.RawArray(ctypes.c_float, total_elements)
arr = np.frombuffer(shared_arr, dtype=np.float32).reshape(shape)
```

<div class="alert alert-block alert-danger">

##### **6.11 Speedup analysis: sequential-parallel approach (EXAM CALCULATION)**

</div>


Before introducing tree reduction, the lecture analyses the simpler "row-sum then aggregate" approach. This derivation appears directly as an exam question (2024R Q18).

**Setup:** sum all $N$ elements in an $n \times n$ matrix ($N = n^2$) using $T$ workers.

**Step 1 - parallel part:** each worker sums $N/T$ rows simultaneously.

**Step 2 - serial part:** aggregate the $T$ partial sums serially.

$$T(T) = \frac{N}{T} + T$$

Speedup vs fully serial ($T(1) = N$):

$$S(T) = \frac{N}{N/T + T}$$

**Maximum speedup** - differentiate with respect to $T$ and set to zero:

Optimal $T = \sqrt{N}$, giving:

$$S_{\max} = \frac{N}{N/\sqrt{N} + \sqrt{N}} = \frac{N}{2\sqrt{N}} = \frac{\sqrt{N}}{2}$$

> KEY RESULT: The sequential-parallel approach gives maximum speedup $\sqrt{N}/2$.

**Comparison with tree reduction** (speedup $N / \log_2(N)$):

For $N = 100$: sequential-parallel $= \sqrt{100}/2 = 5\times$, tree $= 100/\log_2(100) \approx 15\times$

For $N = 10{,}000$: sequential-parallel $= 50\times$, tree $= 10000/13.3 \approx 750\times$

Tree reduction wins by an increasing margin as $N$ grows. The ratio is $\frac{N/\log_2 N}{\sqrt{N}/2} = \frac{2\sqrt{N}}{\log_2 N}$, which grows without bound.

> NOTE: The 2024R exam Q18 asks exactly this: compare the two approaches for summing an $n \times n$ matrix. The answer is speedup of $n/\log_2(n)$ for tree reduction vs $\sqrt{n}/2$ for sequential-parallel.


<img src="SymPyBilleder/2026-05-21-22-04-57.png" width="550">

<div class="alert alert-block alert-danger">

##### **6.12 Row sum vs column sum - even for large N (Q16/Q17 type)**

</div>


When $n$ is large enough that a row/column cannot fit in cache, the question becomes: does cache locality still matter?

**Yes - spatial locality still affects performance even outside cache.**

Summing a row: thread accesses `x[i, 0], x[i, 1], ..., x[i, n-1]` - sequential addresses. Each cache line loaded serves multiple values.

Summing a column: thread accesses `x[0, j], x[1, j], ..., x[n-1, j]` - each element is $n \times \text{itemsize}$ bytes apart. Every access is a cache miss, loading a full 64-byte cache line to use only 1 value.

> KEY: The statement "$n$ is large enough that a row/column cannot fit in cache" means temporal locality is gone, but spatial locality (access pattern within each cache miss) still applies. Row access is always more cache-efficient than column access, regardless of array size.

> THREADING IS APPROPRIATE HERE: `np.sum()` on a row releases the GIL (it is C-level NumPy). So `ThreadPool` with row sums is valid - threads run in parallel. This is the Q17 exception to the GIL rule.

<div class="alert alert-block alert-info">

### **Week 7**

High-Performance Pandas and Apache Arrow
</div>



<div class="alert alert-block alert-danger">

##### **7.1 The core problem with pandas and large data**

</div>


Pandas is the de facto standard for tabular data in Python, but it has a serious memory problem: a CSV file on disk is compact text, but when pandas loads it into RAM, each value becomes a Python object. A 120 MB zip file can balloon to 2 GB in memory. The goal of this week is to understand why this happens and how to fix it.


<div class="alert alert-block alert-danger">

##### **7.2 Measuring DataFrame memory correctly**

</div>

```python
df = pd.read_csv('bighuge.csv.zip')
df.info(memory_usage='deep')           # shows per-column dtype and MB
df.memory_usage(deep=True).sum()       # total bytes as one number
```

> CRITICAL: `deep=True` is essential. Without it, pandas only reports the shallow size of index structures and misses the actual memory consumed by string/object columns. Always use `deep=True` when diagnosing memory.

A useful helper to see all columns at once:

```python
def summarize_columns(df):
    print(pd.DataFrame([
        (c, df[c].dtype, len(df[c].unique()),
         df[c].memory_usage(deep=True) // (1024**2))
        for c in df.columns
    ], columns=['name', 'dtype', 'unique', 'size (MB)']))
    print('Total size:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')
```

This tells us the column name, number of unique values, dtype, and memory. The number of unique values is the key to deciding which reduction strategy to use.

<div class="alert alert-block alert-danger">

##### **7.3 Three strategies for memory reduction**

</div>

The lecture organises this into three techniques, applied in order of impact:

**1. Date types, biggest win for timestamp columns**

Columns read as `object` (strings like `"2023-01-17T23:59:00Z"`) cost around 400–600 MB each because every value is a separate Python string object. Converting to `datetime64` stores the same information as a contiguous array of 8-byte integers:

```python
df['observed'] = pd.to_datetime(df['observed'])
# Before: 600 MB (object)  →  After: 62 MB (datetime64[ns])
```

> KEY INSIGHT: The reason `object` is so expensive is that each element is a pointer to a separately heap-allocated Python object with its own reference count, type pointer, and string data. A `datetime64` array is a flat block of integers - no per-element Python overhead.


**2. Categorical encoding, for columns with few unique values**

If a column has many rows but few unique values, the `category` dtype stores the unique values once in a lookup table and the column as a small integer index array:

```python
df['parameterId'] = df['parameterId'].astype('category')
# 47 unique strings across 8M rows
# Before: hundreds of MB  →  After: ~6 MB
```

This also works for numeric columns. `coordsx`/`coordsy` have only 224/219 unique float values across 8M rows, encoding as category stores those floats once and replaces the column with indices.

> RULE OF THUMB: If `#unique << #rows`, use `category`. The break-even is roughly when `#unique < #rows / 2` for string columns, or when cardinality is very low for numeric columns.



**3. Smaller numeric types, for integer and float columns**

pandas defaults to `float64` (8 bytes) and `int64` (8 bytes) for all numeric columns, even when the data would fit in a smaller type. Always check the actual `min` and `max` before downcasting:

```python
# Integer dtype ranges:
# int8:   -128 to 127
# int16:  -32,768 to 32,767
# int32:  -2,147,483,648 to 2,147,483,647
# int64:  very large

# Float precision check before downcasting:
max_error = (df['value'] - df['value'].astype('float32')).abs().max()
# If max_error is acceptable, use float32 (halves memory)
# If max_error is inf, float16 overflows -  stay at float32 minimum

df['stationId'] = df['stationId'].astype('int16')   # max 32,767 ✓
df['value']     = df['value'].astype('float32')      # acceptable precision ✓
```

> NOTE: Never convert integer columns to float when downcasting, rounding errors can corrupt integer values. Similarly, never convert a numeric column to datetime. Match the reduction to the actual data type.

> CRITICAL TRAP: The number of unique values does NOT determine the correct integer type. `stationId` has only 247 unique values but a max of 34,339 -  this requires `int16`, not `int8`. Always look at `min` and `max`, never `#unique`, when downcasting integers.


**Combined effect on the exercise dataset:**

| Column | Before | After | Technique |
|---|---|---|---|
| `observed` | 600 MB (object) | 62 MB (datetime64) | Date type |
| `parameterId` | around 200 MB (object) | around 6 MB (category) | Categorical |
| `coordsx/y` | around 60 MB (float64) | around 6 MB (category) | Categorical |
| `stationId` | around 60 MB (int64) | around 15 MB (int16) | Smaller int |
| `value` | around 60 MB (float64) | around 30 MB (float32) | Smaller float |

 KEY PRINCIPLE HERE IS THAT pandas' default behaviour on `read_csv` is to use the broadest safe type for each column, which is almost always wasteful. Always profile with `summarize_columns` before committing to a schema.


<div class="alert alert-block alert-danger">

##### **7.4 Pandas chunking for larger-than-memory data**

</div>

If a file is too large to fit in RAM at all, you can process it in chunks:

```python
dfc = pd.read_csv('bighuge.csv', chunksize=100_000)  # returns an iterator
coordsum = 0.0
for df in dfc:
    coordsum += df['coordsx'].sum()
```

Each iteration loads `chunksize` rows, processes them, and discards them before loading the next batch. The full DataFrame never exists in memory simultaneously.

> PRO: Can process files larger than RAM.

> CON: More complex code. Operations that need the full dataset at once (e.g. global sort, median) are harder to implement.


<div class="alert alert-block alert-danger">

##### **7.5 Apache Arrow and why it exists**

</div>

Apache Arrow is a language-independent, columnar memory format implemented in C++. It is not a replacement for pandas, it lacks pandas' rich analytics API. Instead it is used for:

> **Fast CSV reading**, Arrow's CSV reader is multi-threaded C++ vs pandas' single-threaded Python

> **Efficient string storage**, Arrow stores strings as a contiguous byte buffer + offset array, not as Python objects

> **Efficient file formats**, Arrow underpins Parquet, which is far better than CSV for storage

```python
from pyarrow import csv as pa_csv
import pyarrow as pa

table = pa_csv.read_csv('2023_01.csv')
# 3 s (Arrow) vs 11.1 s (pandas) -  3.7 x faster, multi-threaded C++

df = table.to_pandas()
# +0.7 s conversion -  total 3.7 s, still well under 11.1 s
```


**Why the Arrow table is smaller in memory than the equivalent pandas DataFrame:**

PyArrow table: 507 MB. Equivalent pandas DataFrame: 919 MB. Same data. The difference is strings: Arrow stores all strings in one contiguous byte buffer with an offset array pointing to where each string starts and ends. Pandas stores string columns as arrays of Python object pointers, each pointing to a separately allocated Python string with full Python overhead (reference count, type info, hash cache). This per-object overhead dominates for large string columns.




**Applying type conversions on load with Arrow:**

It is always cheaper to load data in the correct type than to load it large and then convert. Arrow supports this natively:

```python
convert_options = pa_csv.ConvertOptions(
    column_types={
        'value':       pa.float32(),
        'parameterId': pa.dictionary(pa.int32(), pa.string()),  # = category
        'coordsx':     pa.dictionary(pa.int32(), pa.float64()),
        'coordsy':     pa.dictionary(pa.int32(), pa.float64()),
    }
)
table = pa_csv.read_csv(fname, convert_options=convert_options)
# Result: 312 MB -  data never existed in its large form in memory
```

`pa.dictionary(index_type, value_type)` is Arrow's equivalent of pandas `category` dtype.

<div class="alert alert-block alert-danger">

##### **7.6 Parquet, the better file format for tabular data**

</div>

CSV is a terrible format for large data: it stores numbers as ASCII text (wasteful), has no schema (types must be inferred on every load), and requires full parsing on every read.

Parquet is a binary, columnar file format designed for large datasets:

```python
# Write once:
df.to_parquet('data.parquet')

# Read (much faster):
df = pd.read_parquet('data.parquet')
```

**Performance comparison for the same dataset:**

| Format | File size | Read time |
|---|---|---|
| CSV (uncompressed) | 700 MB | $12 \: s$ (pandas) |
| CSV (zip) | 120 MB | $11 \: s$ (pandas) |
| Parquet | **86 MB** | **$1 \: s$** (pandas), $0.4 \: s$ (Arrow) |

Parquet achieves this through two mechanisms:

> **Binary encoding**, numbers stored as raw bytes, not ASCII digits. No text parsing needed on read.

> **Column-level compression**, each column is compressed independently (Snappy, Zstd, and such). Homogeneous typed data in a single column compresses much better than mixed-type row data in CSV.

> KEY INSIGHT: The read speedup ($10\times$) comes almost entirely from eliminating text parsing. Reading raw binary integers from disk is trivial; parsing "1234.5678" as a float from ASCII characters is not.

<div class="alert alert-block alert-danger">

##### **7.7 Decision guide: when to use what**

</div>

| Situation | Strategy |
|---|---|
| String column with timestamps | Convert to `datetime64` |
| String column with few unique values | Convert to `category` |
| Numeric column with small range | Downcast to `int16`/`int32`/`float32` |
| File too large to fit in RAM | `pd.read_csv(chunksize=N)` |
| Need fast CSV loading | Use `pyarrow.csv.read_csv` then `.to_pandas()` |
| Loading same data repeatedly | Convert to Parquet once, read Parquet thereafter |
| Need to apply types on load | Use Arrow `ConvertOptions` |

<div class="alert alert-block alert-danger">

##### **7.8 Optimizing data loading time**

(book notes)

</div>

Examples the book mentions are converting to date time, changing floats from for instance 64 to 32, and using categorical data types instead of strings.

```python
def reduce_dmi_df(df):
    df['parameterId'] = df['parameterId'].astype('category')
    df["created"] = pd.to_datetime(df["created"], format="ISO8601")
    df["observed"] = pd.to_datetime(df["observed"], format="ISO8601")
    df["coordsx"] = df["coordsx"].astype('float32')
    df["coordsy"] = df["coordsy"].astype('float32')
    df["value"] = df["value"].astype('float32')
    df["stationId"] = df["stationId"].astype('int32')
    return df
```

<div class="alert alert-block alert-danger">

##### **7.9 Indexing data**

(book notes)

</div>

General rule of thumb is that row-based analysis should usually be approached in a declarative way, meaning vectorizing if possible but at least avoid explicit iteration.

<div class="alert alert-block alert-danger">

##### **7.10 Explicit use of NumPy**

(book notes)

</div>

If we write `to_numpy` we return a reference to the underlying pandas array.

We can also do `to_numpy(copy=True)` to get a copy to make changes that are not reflected on the original. But copying will double the memory usage and there will be a time cost too.

<div class="alert alert-block alert-danger">

##### **7.11 Exam skill: reading a `summarize_columns` table and choosing the right strategy**

</div>

**Integer type ranges (memorise these tables):**

**Integers (signed):**
| Type | Min | Max |
|---|---|---|
| int8 | $-128$ | $127$ |
| int16 | $-32,768$ | $32,767$ |
| int32 | $-2,147,483,648$ | $2,147,483,647$ |
| int64 | $-9.22 \times 10^{18}$ | $9.22 \times 10^{18}$ |

**Integers (unsigned):**
| Type | Min | Max |
|---|---|---|
| uint8 | $0$ | $255$ |
| uint16 | $0$ | $65,535$ |
| uint32 | $0$ | $4,294,967,295$ |
| uint64 | $0$ | $1.84 \times 10^{19} $

**Floats:**
| Type | Resolution | Min/Max |
|---|---|---|
| float16 | $0.001$ | $\pm 65,504$ |
| float32 | $1e-6$ | $\pm 3.40 \times 10^{38}$ |
| float64 | $1e-15$ | $\pm 1.80 \times 10^{308}$ |


When given a table like this in the exam:

| Col. name | dtype | #unique | min | max | size |
|---|---|---|---|---|---|
| filename | object | 487M | -  | -  | 21.3 GB |
| version | int64 | 43 | 0 | 42 | 2.1 GB |
| sizediff_kb | int64 | 4366 | -12 | 4353 | 2.1 GB |

Work through each column with this decision tree:

**Step 1 - What is the current dtype?**

> `object` (string): go to Step 2a.

> numeric (`int64`, `float64`): go to Step 2b.

**Step 2a - For `object` columns:**

> Does the example value look like a timestamp/date? → `pd.to_datetime()` (biggest win, $\sim 10 x$ smaller)

> Is `#unique << #rows`? → `.astype('category')` (stores unique strings once)

> Otherwise: cannot reduce meaningfully

**Step 2b - For numeric columns:**

> Is it integer data? → find smallest int type where `min ≥ type_min AND max ≤ type_max`

> Is it float data? → test `(col - col.astype('float32')).abs().max()` -  if not `inf`, use `float32`

> NEVER convert integer to float -  rounding corrupts values

> NEVER convert numeric to datetime -  nonsensical



**Worked example (2025 exam Q17):**

`version`: int64, 43 unique values, min=0, max=42. The max ($42$) fits in `int8` (max $127$) or `uint8` (max $255$). Use `int8`. Answer: **C - convert to a smaller integer type**.

`sizediff_kb`: int64, min=-12, max=4353. Can't use `int8` (max $127$). Can't use `uint8` (negative values). Max $4353$ < $32,767$ so fits in `int16`. Answer: **convert to `int16`**.

`filename`: object, 487M unique values -  too many for `category` (nearly one unique per row). No timestamp pattern. Answer: **cannot meaningfully reduce**.

> KEY TRAP: A column can be integer-typed but still not reducible if the value range is too large. Always check BOTH `min` and `max` against the type range, not just the number of unique values.

> KEY TRAP: The 2025 exam Q18 shows `fname_hash` as `int64` with 487M unique values and min/max in the millions - the values are too large for any smaller integer type. Answer: **cannot reduce**.

<div class="alert alert-block alert-info">

### **Week 8**

Storing Big Data
</div>



<div class="alert alert-block alert-danger">

##### **8.1 The problem: files larger than RAM**

</div>

Weeks 3–7 focused on what happens in memory. Week 8 asks: what if the data does not even fit in RAM? Two tools address this: `np.memmap` (memory mapping) and Zarr (chunked, compressed, N-dimensional arrays). They solve the same problem from different angles.



<div class="alert alert-block alert-danger">

##### **8.2 Memory mapping, `np.memmap`**

</div>

Memory mapping is an OS-level (OS stands for operating system) mechanism that lets you treat a file on disk as if it were a NumPy array in RAM. The CPU sees it as normal memory, but the backing store is on disk.

```python
import numpy as np

# Create a new memory-mapped file (mode='w+' = create and read/write)
mm = np.memmap('tmp.raw', mode='w+', shape=(10, 10), dtype='float64')
mm[:5, :5] = 1   # writes directly to disk -  no explicit save needed!

# Re-open an existing file (must specify dtype and shape manually)
mm = np.memmap('tmp.raw', mode='r+', dtype='float64', shape=(10, 10))
```

**Mode options:**

> `'r'` is read-only

> `'r+'` is read/write (file must already exist)

> `'w+'` is create new or overwrite (read/write)

> CRITICAL:

If we open without specifying `dtype` and `shape`, NumPy defaults to `uint8` and reads the raw bytes. A $10\times 10$ float64 array = 800 bytes, so it will appear as shape `(800,)` of uint8. Always specify both.

> CRITICAL:

Changes are written to disk immediately -  there is no explicit save step. This means modifying a memmap modifies the file. Be careful with `mode='r+'` on important data.

**How it works under the hood:**


The OS maps the file into the process's virtual address space, the "fake memory" region shown in the Week 3 diagram. When code reads `mm[i, j]`, the OS transparently fetches the relevant page from disk (a page fault) and puts it in RAM. Dirty pages are flushed back to disk when evicted or when the program exits. This means:

1. Data is loaded **on demand** from disk, only accessed pages are ever in RAM
2. The array can be **larger than physical RAM**, the OS manages what fits in RAM at any time
3. It can be treated **like a normal NumPy array**, same indexing, slicing, arithmetic


**Parallel writing with memmap:**

Multiple processes can safely write to non-overlapping slices simultaneously because each slice maps to different pages on disk. No locks needed. This makes memmap a good backend for parallel Mandelbrot computation where each process writes its own rows.

```python
# Process i writes its rows
mm = np.memmap('output.raw', mode='r+', dtype='int32', shape=(N, N))
mm[row_start:row_end, :] = local_result
```

**Downsampling with memmap:**

```python
mm = np.memmap('mandelbrot.raw', mode='r', shape=(N, N), dtype='int32')
downsampled = mm[::step, ::step]   # only touched pages are loaded from disk
```

With `step=16`, only 1/256th of the data is ever loaded. This is much more efficient than `np.load` which loads the entire file even if we only need part of it.

<div class="alert alert-block alert-danger">

##### **8.3 Zarr - chunked, compressed, N-dimensional arrays**

</div>

Zarr is the modern answer to HDF5 for storing large NumPy arrays. It stores arrays as directories of compressed chunk files, rather than one monolithic file.

```python
import zarr

# Create a new Zarr array
store = zarr.open('mandelbrot.zarr', mode='w',
                  shape=(1000, 1000),
                  dtype='int32',
                  chunks=(200, 200))   # 25 chunks total

# Write to it like a NumPy array
store[:200, :200] = data

# Inspect it
store.info
```

**Reading the `array.info` output**, this appears directly in the exam slides so know how to interpret every field:

```
Type          : zarr.core.Array
Data type     : int32
Shape         : (1000, 1000)      ← full array dimensions
Chunk shape   : (200, 200)        ← each chunk is 200 x200 elements
Order         : C                 ← row-major (C-order) within each chunk
Read-only     : False
Compressor    : Blosc(cname='lz4', clevel=5, ...)   ← compression algorithm
Store type    : zarr.storage.DirectoryStore
No. bytes         : 4000000  (3.8 MB)   ← uncompressed size
No. bytes stored  : 95666   (93.4 KB)   ← actual disk usage after compression
Storage ratio     : 41.8    ← how many times smaller the stored version is
Chunks initialized: 25/25   ← all 25 chunks have been written
```

> NOTE: `No. bytes / No. bytes stored = Storage ratio`. A ratio of 41.8 means the stored version is $41.8 \times$ smaller than the raw uncompressed array. High ratios occur when data has patterns (e.g. large regions of the Mandelbrot array are constant outside the set).



**Zarr's internal structure and why it beats HDF5:**

```bash
$ ls mandelbrot.zarr/
0.0  0.1  0.2  0.3  0.4   ← chunk files for row 0
1.0  1.1  1.2  1.3  1.4   ← chunk files for row 1
...
.zarray                   ← metadata file (shape, dtype, chunks, compressor)
```

Each chunk is a **separate file**. This is the key architectural difference from HDF5 (which uses a single file). Because chunks are separate files, **multiple processes can read and write different chunks simultaneously**, no file locking, no serialisation. HDF5 does not support concurrent writes.

<div class="alert alert-block alert-danger">

##### **8.4 Chunk size and shape, the most important tuning parameter**

</div>

The chunk size controls the trade-off between speed and storage. From the exercise results on a $1000 \times 1000$ Mandelbrot array:

| Chunk size | Runtime | Stored size |
|---|---|---|
| 10 | $24.2 \: s$ | $285 \: MB$ |
| 25 | $3.6 \: s$ | $38 \: MB$ |
| 50 | $1.1 \: s$ | $9.5 \: MB$ |
| **100** | **$1.3 \: s$** | **$2.4 \: MB$** |
| 200 | $4.3 \: s$ | $656 \: KB$ |

**Why too small is slow:** too many chunks → many tiny files → high filesystem overhead, and parallelism overhead from spawning a process per chunk dominates.

**Why too large is also slow:** with a $1000 \times 1000$ array and chunk size 200, there are only 25 chunks. With $24+$ workers, each gets at most one chunk, but many workers sit idle if chunks vary in compute time. Load balancing breaks down with too few chunks.

**Why storage decreases monotonically:** larger chunks include more of the large uniform-value regions outside the Mandelbrot set, which compress to almost nothing. Tiny chunks waste space on per-chunk metadata headers.

> RULE: Chunks should be **at least 1 MB uncompressed** as a starting point. Match chunk shape to your access pattern.

**Chunk shape must match access pattern:**

The lecture diagram shows three cases:

> **Tall narrow chunks**, good for column-wise access (read one column → touches few chunks)

> **Short wide chunks**, good for row-wise access (read one row → touches few chunks)

> **Square chunks**, good for neighbourhood access (e.g. image processing, convolution)

If the chunk shape does not match the access pattern, every access loads a full chunk but uses only a fraction of it, the same waste as column-wise access in a row-major NumPy array.


<div class="alert alert-block alert-danger">

##### **8.5 Zarr vs memmap, when to use which**

</div>




| Feature | `np.memmap` | Zarr |
|---|---|---|
| Format | Raw binary (no compression) | Chunked + compressed |
| File structure | Single file | Directory of chunk files |
| Compression | None | Built-in (lz4, blosc, gzip, etc.) |
| Parallel writes | ✓ (non-overlapping slices) | ✓ (separate chunk files) |
| HDF5 compatibility | ✗ | ✗ (separate format) |
| Larger-than-RAM | ✓ | ✓ |
| Metadata stored | ✗ (must remember dtype/shape) | ✓ (`.zarray` file) |
| Best for | Simple raw arrays, fast writes | Large arrays needing compression and parallel I/O |


KEY NOTE from exercise results: memmap at $5.3 \: MB$ raw and $0.8 \: s$ vs Zarr at chunk = 100 giving $2.4 \: MB$ and $1.3 \: s$. Zarr uses less than half the disk space at only a small runtime cost, and this is a compelling trade-off as data sizes grow.


<div class="alert alert-block alert-danger">

##### **8.6 Zarr storage backends**

</div>

Zarr supports multiple backends beyond the local filesystem:

```python
zarr.open('data.zarr')                    # DirectoryStore - default, local disk
zarr.open(zarr.MemoryStore())             # MemoryStore - everything in RAM
zarr.open('s3://bucket/data.zarr')        # Amazon S3
zarr.open('gs://bucket/data.zarr')        # Google Cloud Storage
```

This makes Zarr cloud-native, this is the same code that writes to a local directory can write to cloud storage with a URL change.


<div class="alert alert-block alert-danger">

##### **8.7 Practical cheat sheet**

</div>

```python
# memmap - create
mm = np.memmap('file.raw', mode='w+', shape=(N, M), dtype='float64')

# memmap - open existing
mm = np.memmap('file.raw', mode='r+', dtype='float64', shape=(N, M))

# Zarr - create
z = zarr.open('file.zarr', mode='w', shape=(N, M), dtype='int32',
              chunks=(C, C))

# Zarr - open existing
z = zarr.open('file.zarr', mode='r+')

# Zarr - inspect
z.info                          # full metadata summary
z.nbytes                        # uncompressed bytes
z.nbytes_stored                 # bytes actually on disk
z.nbytes / z.nbytes_stored      # storage ratio (same as z.info shows)
```


<div class="alert alert-block alert-danger">

##### **8.8 Memory mapping**

(book notes)

</div>

It occurs when a part of the memory is directly associated with a part of the filesystem.

In case of NumPy, an array that is persisted to storage can be assed with the nurmal NumPy API, and NumPy will take care of bringing to RAM whatever parts we need from the array.

<div class="alert alert-block alert-danger">

##### **8.9 NumPy copy-on-write**

(book notes)

</div>

NumPy memory mapping allows us to use copy-on-write, which permits us to have several copies of a disk array loaded into memory and pay a lower price in terms of memory usage.

<div class="alert alert-block alert-danger">

##### **8.10 Chunking**

(book notes)

</div>

It means processing a file in chunks so in parts. For instance like:
```python
import pandas a pd
table_chunks = pd.read_csv('big_file.csv', chunksize=1000)

print(type(table_chunks))

for chunk in table_chunks:
    print(chunk.shape)
```
So just add parameter `chunksize` to the `read_csv` function, and it will return an iterator that yields chunks of the file as DataFrames. In this case, each chunk will contain 1000 rows.

<div class="alert alert-block alert-danger">

##### **8.11 Parquet file**

(book notes)

</div>

Parquet allows us to read row group by row group like:
```python
pf = pq.ParquetFile('big_file.parquet')

for groupi in range(pf.num_row_groups):
    group = pf.read_row_group(i)
    print(type(group), len(group))
    break
```
This is one of the ways to read the Parquet data.

<div class="alert alert-block alert-danger">

##### **8.12 Zarr**

(book notes)

</div>

Zarr allows us to efficiently store homogeneous multidimensional arrays with different backends and different encoding formats.

When Zarr reads a file, it returns a `Group` object, and the `groups` method will return a generator with all the subgroups inside, so we can rely on that to traverse a Zarr repository.

<div class="alert alert-block alert-danger">

##### **8.13 EXAM SKILL: reasoning through chunk shape questions**

</div>


This is the most tested concept in Week 8 and the one that requires actual reasoning, not just memorising the three diagrams.

**The method - 4 steps:**

**Step 1:** Read the access pattern from the code. What does one iteration of the loop access?

```python
for i in range(a.shape[0]):      # i iterates over rows
    s[i] = np.sum(a[i])          # accesses row i → one full row per iteration
```

**Step 2:** For each candidate chunk shape, count how many chunks are loaded to satisfy one iteration.

**Step 3:** The shape that minimises chunk loads per iteration wins.

**Step 4:** To calculate chunk loads: if the access is one full row of a `(1000, 1000)` array, and a chunk has shape `(C_rows, C_cols)`, then accessing one row loads `1000 / C_cols` chunks (one per horizontal tile that row spans).


**Worked example 1 - 2024R exam Q5**

```python
def process(fname, columns):
    x = zarr.open(fname, mode='r')    # array shape: 1000  x 100000
    s = 0
    for i in columns:
        s += x[:, i].sum()            # accesses column i - all 1000 rows, 1 column
    return s
```

Access pattern: **one full column per iteration** (`x[:, i]`).

Array shape: `1000  x 100000`. Candidate shapes: `(a) 10 x10000`, `(b) 100 x1000`, `(c) 1000 x100`.

For column `i`, the chunks that contain it are those whose column range covers `i`. How many chunks does a single column span?

> `(a) 10 x10000`: each chunk is 10 rows tall and 10000 columns wide. One column spans `1000/10 = 100` chunks (100 chunks stacked vertically).

> `(b) 100 x1000`: each chunk is 100 rows tall and 1000 columns wide. One column spans `1000/100 = 10` chunks.

> `(c) 1000 x100`: each chunk is 1000 rows tall and 100 columns wide. One column spans `1000/1000 = 1` chunk - the entire column fits in one chunk!

**Answer: (c) `1000 x100`** - one chunk load per column access. ✓

> KEY INSIGHT: For column access, you want the chunk to be as tall as possible (cover the full column height) and as narrow as possible. This minimises the number of chunks a single column spans.

**Worked example 2 - 2025 exam Q20**

```python
def process(a):
    s = np.zeros(a.shape[0])
    for i in range(a.shape[0]):
        s[i] = np.sum(a[i])         # accesses row i -  1 row, all 1024 columns
    return s
```

Array shape: `1024  x 1024`. Candidate shapes: `(a) (1, 1024)`, `(b) (1024, 1)`, `(c) (32, 32)`.

For row `i`, the chunks that contain it are those whose row range covers `i`. How many chunks does a single row span?

- `(a) (1, 1024)`: chunk is 1 row tall, 1024 columns wide. Row `i` spans exactly `1024/1024 = 1` chunk.
- `(b) (1024, 1)`: chunk is 1024 rows tall, 1 column wide. Row `i` spans `1024/1 = 1024` chunks.
- `(c) (32, 32)`: chunk is 32 rows tall, 32 columns wide. Row `i` spans `1024/32 = 32` chunks.

**Answer: (a) `(1, 1024)`** - one chunk load per row access. ✓

> KEY INSIGHT: For row access, you want the chunk to span the full row width. Shape `(1, 1024)` means each chunk *is* one full row -  accessing any row loads exactly 1 chunk.


**The general rule:**

> Match the chunk's "long dimension" to the access's "long dimension". If you access full rows, make chunks wide. If you access full columns, make chunks tall. If you access 2D blocks, use square chunks.



**How to calculate chunk memory size**

From 2024R Q6: given chunk shape `(1000, 100)` and dtype `float64` (8 bytes):

$$\text{chunk bytes} = 1000 \times 100 \times 8 = 800{,}000 \text{ bytes} = \frac{800{,}000}{1024} \approx 781 \text{ KB}$$

General formula:
$$\text{chunk bytes} = \prod_{\text{dim}} \text{chunk\_size}_{\text{dim}} \times \text{itemsize}$$

> NOTE: This is the *uncompressed* size - what is loaded into RAM when you read a chunk. The *stored* size on disk will be smaller due to compression (shown as `No. bytes stored` in `array.info`).


**The `array.cdata_shape` attribute**

```python
>>> array = zarr.open('mandelbrot.zarr')
>>> array.shape         # (1000, 1000)  ← full array
>>> array.chunks        # (200, 200)    ← each chunk
>>> array.cdata_shape   # (5, 5)        ← number of chunks per dimension
```

`cdata_shape` tells you the grid of chunks: a `(1000, 1000)` array with `(200, 200)` chunks has `(5, 5) = 25` total chunks. This is what the `Chunks initialized: 25/25` line in `array.info` refers to.

Total chunks = `array.cdata_shape[0]  x array.cdata_shape[1]  x ...`

<div class="alert alert-block alert-danger">

##### **8.14 The 3-step method for "max chunk size given RAM limit"**

</div>


**Step 1 - bytes per row:**

Count the columns, check their dtypes, multiply:

```
3 columns × int64 (8 bytes each) = 24 bytes per row
```

(EXAMPLE FROM 2025 EXAM Q18)

**Step 2 - convert RAM limit to bytes:**

```
24 MB × 1,000,000 = 24,000,000 bytes
```

(the exam uses 1 MB = 1,000,000 - that's what the official solution uses too)

**Step 3 - max rows = RAM bytes ÷ bytes per row:**

```
24,000,000 ÷ 24 = 1,000,000 rows
```

→ Answer A: 800,000 is the only option ≤ 1,000,000. ✓

##### **Exact chunk load counting (exam calculation)**


The exam gives you an array shape, a chunk shape, and an access pattern, then asks how
many chunks are loaded. The method is always the same four steps.

**Formula:** for each dimension, `ceil(access_size / chunk_size)` gives the number of
chunks touched in that dimension. Multiply across dimensions.

$$
\text{chunks loaded} = \prod_{\text{dim } d} \left\lceil \frac{\text{access size in dim } d}{\text{chunk size in dim } d} \right\rceil
$$

**Example 1  - reading one full row:**

Array shape `(1024, 1024)`, chunk shape `(32, 32)`, access = row `i` = `arr[i, :]`.

- Dim 0 (rows): access size = 1, chunk size = 32 → $\lceil 1/32 \rceil = 1$ chunk
- Dim 1 (cols): access size = 1024, chunk size = 32 → $\lceil 1024/32 \rceil = 32$ chunks
- Total: $1 \times 32 = \mathbf{32}$ chunk loads

**Example 2  - reading one full column:**

Same array and chunk shape, access = column `j` = `arr[:, j]`.

- Dim 0 (rows): access size = 1024 → $\lceil 1024/32 \rceil = 32$ chunks
- Dim 1 (cols): access size = 1 → $\lceil 1/32 \rceil = 1$ chunk
- Total: $32 \times 1 = \mathbf{32}$ chunk loads

(Same cost  - both are bad. Square chunks are equally bad for both; tall/narrow chunks
would be great for column access but terrible for row access.)

**Example 3  - reading a small patch:**

Array shape `(1024, 1024)`, chunk shape `(256, 32)`, access = `arr[0:100, 0:100]`.

- Dim 0: $\lceil 100/256 \rceil = 1$ chunk
- Dim 1: $\lceil 100/32 \rceil = 4$ chunks
- Total: $1 \times 4 = \mathbf{4}$ chunk loads

**Shortcut:** if the access exactly aligns with chunk boundaries (e.g. reading exactly one
chunk's worth of rows), no ceiling rounding is needed. The exam often uses numbers that
divide cleanly to test whether you know the formula, not arithmetic.

> KEY RULE: minimise chunks loaded = match chunk shape to access pattern. A chunk that
> spans the full access dimension in one direction costs only 1 load in that dimension.

<div class="alert alert-block alert-info">

### **Week 9**

Numba and GPU Computing
</div>

<div class="alert alert-block alert-danger">

##### **9.1 Numba on the CPU**

</div>

Before moving to the GPU, Week 9 starts with Numba as a CPU accelerator. The key insight is that Python loops are slow not because of the arithmetic but because of the interpreter overhead - type checking, object lookups, and reference counting on every single iteration.

**`@jit(nopython=True)`, JIT compiling Python loops:**

```python
from numba import jit

@jit(nopython=True)
def matmul_jit(A, B, C):
    for i in range(A.shape[0]):
        for k in range(A.shape[1]):
            for j in range(B.shape[1]):
                C[i, j] += A[i, k] * B[k, j]
```

> The `@jit` decorator tells Numba to compile this function to native machine code the **first time it is called**.

> `nopython=True` forces Numba to compile **entirely without the Python interpreter**, if Numba cannot figure out the types, it fails loudly rather than silently falling back to slow Python. Always use `nopython=True`.

> **The first call is slow**, that is when compilation happens. Always run once before timing to "warm up" the JIT, then time the second call.

> The 470 times speedup over pure Python loops comes from eliminating all interpreter overhead, the compiled output is equivalent to what a C compiler would produce.

> KEY NOTE: When `dot_jit` is slower than `np.dot` on the slides, this is expected. `np.dot` calls a highly optimised BLAS routine (DGEMM) that uses SIMD vector instructions, cache tiling, and decades of tuning. Numba JIT produces generic machine code without those tricks. Numba wins when we need to JIT-compile custom loops that NumPy cannot express.

**Loop re-ordering with Numba**, because Numba compiles to actual machine code, the `i → k → j` cache-friendly loop order matters just as much as in Week 3. The Numba + cache-friendly order gives an additional around 6 times over Numba + cache-unfriendly order.


<div class="alert alert-block alert-danger">

##### **9.2 GPU architecture**

</div>

**The GPU is a co-processor.** The CPU (host) drives all computation. The GPU (device) is called by the CPU to run massively parallel work. They have **separate memory spaces** connected by the PCIe bus.

```
CPU (host)          PCIe bus          GPU (device)
[host memory]  ←────────────────→  [device memory]
```

Data transfer over PCIe is expensive -  this is the central performance challenge of GPU programming.

**GPU hardware hierarchy:**

```
GPU
└── Many Streaming Multiprocessors (SMs)
    └── Each SM contains many CUDA cores (streaming processors)
        └── Each SM has its own L1 cache shared by all cores in the SM
└── L2 cache shared by all SMs
└── GPU main memory (DRAM)
```

A typical GPU has thousands of CUDA cores across dozens of SMs. All cores run at a lower clock speed than CPU cores, but their sheer number enables massive parallelism.

**CPU vs GPU, the bus metaphor from the book:**

> A CPU is like a Ferrari - fast for a few passengers. A GPU is like a bus - slow per seat, but can carry 500 people at once. If you only need to process one element, the CPU wins. If you need to process millions of elements independently, it is not even a fair competition.


<div class="alert alert-block alert-danger">

##### **9.3 The CUDA programming model**

</div>

**The kernel function** is the GPU entry point, the code that every thread executes:

```python
from numba import cuda

@cuda.jit
def double_kernel(x, y):
    i = cuda.grid(1)       # this thread's global index
    y[i] = 2 * x[i]       # each thread handles one element
```

Key rules about kernels:

> Decorated with `@cuda.jit` - Numba compiles this to CUDA code

> **Cannot return values** - results must be written into an output array passed as an argument

> Every thread runs the **same code** but on a different element (SIMT model)

> The mental model: **one thread per output element**



**The thread grid - threads → blocks → grid:**

```
Grid
└── Many blocks (blocks per grid, bpg)
    └── Each block contains many threads (threads per block, tpb)
        └── All threads in a block run on the same SM
```

```python
tpb = 512           # threads per block (max 1024)
bpg = len(x) // tpb  # blocks per grid

double_kernel[bpg, tpb](x, y)   # launch kernel
#             ^    ^
#             |    threads per block
#             blocks per grid
```

**Thread identity inside a kernel:**

```python
tid = cuda.threadIdx.x              # local index within this block (0 to tpb-1)
bid = cuda.blockIdx.x               # which block am I in
i   = cuda.grid(1)                  # global index = bid * tpb + tid
# cuda.grid(1) is shorthand for the above
```

**Bounds check - always needed when $n$ is not divisible by tpb:**

```python
# Ceiling division -  standard pattern:
def get_bpg(n, tpb):
    return (n + tpb - 1) // tpb

@cuda.jit
def double_kernel(x, y):
    i = cuda.grid(1)
    if i < len(x):           # bounds check -  launched extra threads do nothing
        y[i] = 2 * x[i]
```

> RULE: Always use ceiling division `(n + tpb - 1) // tpb` for `bpg`, and always add a bounds check `if i < n` inside the kernel.

<div class="alert alert-block alert-danger">

##### **9.4 Memory transfers (the most important performance topic)**

</div>

The three memory scenarios from the vector addition exercise reveal where time actually goes:

**Scenario 1 - Plain NumPy arrays (default, ~6 ms):**
```python
x = np.random.rand(n)
y = np.empty_like(x)
double_kernel[bpg, tpb](x, y)   # Numba auto-transfers on every call
```
Numba automatically copies `x` to GPU memory before the kernel and copies `y` back after. This happens on **every call** inside the timing loop, measuring transfer + compute + transfer, 200 times.



**Scenario 2 - Pinned memory (around 3.5 ms):**
```python
with cuda.pinned(x):
    double_kernel[bpg, tpb](x, y)
# or allocate pinned directly:
xp = cuda.pinned_array(n)
```
Pinned (page-locked) memory cannot be swapped to disk by the OS. This allows the GPU's DMA engine to transfer data directly without CPU involvement, roughly halving transfer time. Still transferring on every call, just faster.


**Scenario 3 - GPU-resident arrays (around 0.042 ms, 140 times faster):**

```python
d_x = cuda.to_device(x)          # transfer once before timing
d_y = cuda.device_array_like(d_x)

for _ in range(200):
    double_kernel[bpg, tpb](d_x, d_y)   # zero transfers inside loop

y = d_y.copy_to_host()           # transfer back once
```
Arrays live on the GPU the whole time. Inside the loop there is zero transfer, just kernel computation. The 140 times speedup over pinned memory reveals that around $98.8\%$ of the time in Scenario 1 was memory transfer and only $1.2\%$ was actual computation.

> KEY INSIGHT: For work-light kernels (one operation per element like vector addition), PCIe transfer completely dominates. The solution is either to minimise transfers (keep data on GPU) or to use the GPU only for compute-heavy kernels where the arithmetic justifies the transfer cost.

**Arithmetic intensity** = FLOPs per byte transferred. Vector addition has intensity 0.06 (1 FLOP per 16 bytes). Matrix multiply with $N \times N$ matrices has intensity around $N/2$ -  for $N=1024$, that is 512 FLOPs per byte. This is why only $17\%$ of matmul time is transfer vs $99\%$ for vector addition.


<div class="alert alert-block alert-danger">

##### **9.5 Warp divergence and coalesced memory access**

</div>




**Warps:**
The GPU executes threads in groups of 32 called *warps*. All 32 threads in a warp execute the same instruction simultaneously (SIMT - Single Instruction, Multiple Threads). This is what gives GPUs their throughput.

**Warp divergence:**
If threads in the same warp take different branches (e.g. an `if/else` based on thread
index), the GPU must execute *both* branches serially - threads not taking a branch are
masked off (idle). A warp with a 50/50 split runs at half throughput.

```python
@cuda.jit
def divergent_kernel(x, y):
    i = cuda.grid(1)
    if i % 2 == 0:       # half the warp goes here
        y[i] = x[i] * 2
    else:                # other half goes here - both branches execute serially
        y[i] = x[i] + 1
```

**How to avoid it:** restructure so all threads in a warp take the same branch, or use
arithmetic tricks to avoid branching entirely:

```python
# No divergence - same operation for all threads
y[i] = x[i] * 2 * (i % 2 == 0) + (x[i] + 1) * (i % 2 != 0)
```

**Coalesced memory access:**
A warp accesses memory most efficiently when consecutive threads read consecutive
memory addresses - this is called *coalesced* access. The GPU can then satisfy all 32
reads in a single memory transaction.

```python
# Coalesced - thread i reads x[i], so warp reads x[0..31] in one transaction
y[i] = x[i] * 2

# Non-coalesced - thread i reads x[i * stride], addresses are spread out
y[i] = x[i * 32] * 2   # 32 separate memory transactions
```

**Why it matters:** non-coalesced access can reduce memory bandwidth by 32x. For
memory-bound kernels (most are), this is the dominant performance factor.

> RULE: design kernels so that `cuda.grid(1)` maps directly to consecutive array indices.
> Avoid strided access patterns inside kernels.

**Summary table:**

| Concept | Cause | Cost | Fix |
|---|---|---|---|
| Warp divergence | `if/else` on thread index | Up to 32x slowdown | Uniform branching or arithmetic |
| Non-coalesced access | Strided/random index pattern | Up to 32x bandwidth loss | Consecutive thread → consecutive address |
| Thread oversubscription | Total threads > physical cores | Context-switch overhead | Keep total threads ≤ core count |

<div class="alert alert-block alert-danger">

##### **9.6 Warps and performance**

</div>


Threads within a block are executed in groups of 32 called **warps**. All threads in a warp execute the same instruction simultaneously (SIMT). This has two performance implications:

**Warp divergence - avoid if/else that splits a warp:**
```python
# BAD: half the warp takes one branch, half takes the other
# The GPU must serialise both branches
if i % 2 == 0:
    y[i] = x[i] * 2
else:
    y[i] = x[i] + 1
```
When threads in the same warp take different branches, CUDA executes both branches serially with half the threads inactive each time, halving throughput.

**Coalesced memory access, warps should read consecutive memory:**
All threads in a warp read at the same time. If they read consecutive addresses (e.g. `x[i]`, `x[i+1]`, ..., `x[i+31]`), the GPU loads a single 128-byte cache line serving the whole warp. If they read scattered addresses, each thread needs a separate load, around $32 \times$ slower.

> L1 cache line on GPU: 128 bytes = 32 float32 values. One load per warp when access is sequential.


<div class="alert alert-block alert-danger">

##### **9.7 2D kernels for matrix multiplication**

</div>

For a 2D problem like matrix multiplication, use a 2D grid:

```python
@cuda.jit
def matmul_kernel(A, B, C):
    i, j = cuda.grid(2)                    # 2D thread position
    if i < C.shape[0] and j < C.shape[1]:  # bounds check
        tmp = float32(0.)
        for k in range(A.shape[1]):
            tmp += A[i, k] * B[k, j]
        C[i, j] = tmp                      # write once at end

# Launch with 2D grid:
threadsperblock = (16, 16)                 # 256 threads per block total because 16 x 16 = 256
blockspergrid = (math.ceil(N/16), math.ceil(N/16))
matmul_kernel[blockspergrid, threadsperblock](d_A, d_B, d_C)
```

Key points:

> `cuda.grid(2)` returns `(i, j)`, the 2D position in the grid

> Accumulate into a local `tmp` variable, not `C[i, j]` directly, avoids repeated slow global memory writes inside the loop

> Each thread computes one full dot product (N FLOPs), high arithmetic intensity, so transfer cost is only around $17\%$ of total


<div class="alert alert-block alert-danger">

##### **9.8 Week 9 cheat sheet**

</div>


```python
from numba import jit, cuda
import numpy as np

# CPU JIT
@jit(nopython=True)
def my_func(x):
    ...
my_func(x)   # warm up (compiles here)
my_func(x)   # now time this

# GPU kernel - 1D
@cuda.jit
def kernel_1d(x, y):
    i = cuda.grid(1)
    if i < len(x):
        y[i] = x[i] * 2

tpb = 512
bpg = (n + tpb - 1) // tpb   # ceiling division
kernel_1d[bpg, tpb](x, y)

# GPU kernel - 2D
@cuda.jit
def kernel_2d(A, C):
    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        C[i, j] = A[i, j] * 2

tpb2 = (16, 16)
bpg2 = (math.ceil(N/16), math.ceil(N/16))
kernel_2d[bpg2, tpb2](A, C)

# Manual memory transfer
d_x = cuda.to_device(x)          # CPU → GPU
d_y = cuda.device_array(n, dtype=np.float32)  # allocate on GPU
kernel[bpg, tpb](d_x, d_y)
y = d_y.copy_to_host()           # GPU → CPU

# Pinned memory
xp = cuda.pinned_array(n)        # allocate pinned
with cuda.pinned(x):             # pin existing array
    ...

# Thread identity inside kernel
tid = cuda.threadIdx.x   # local index in block
bid = cuda.blockIdx.x    # block index
i   = cuda.grid(1)       # global index (= bid * blockDim.x + tid)
cuda.syncthreads()       # barrier -  all threads in block wait here
```

<div class="alert alert-block alert-danger">

##### **9.9 Automatic vs manual memory transfers (critical exam pattern)**

</div>


When you call a CUDA kernel with plain NumPy arrays, Numba **automatically transfers every argument to the GPU before the kernel and back to the CPU after**. This is convenient but wasteful:

```python
# BAD: Numba auto-transfers x and y on EVERY call
x = np.zeros(1024)
y = np.empty_like(x)
double_kernel[bpg, tpb](x, y)   # transfers x HtoD, y HtoD, then y DtoH
double_kernel[bpg, tpb](y, y)   # transfers y HtoD again, then y DtoH again
```

This means calling a kernel twice on NumPy arrays performs **4 HtoD + 4 DtoH** transfers - most of which are wasted. The optimal approach is to transfer once:

```python
# GOOD: transfer once, run many times, transfer back once
d_x = cuda.to_device(x)               # 1 HtoD
d_y = cuda.device_array_like(d_x)     # allocate on GPU, no transfer
double_kernel[bpg, tpb](d_x, d_y)     # 0 transfers
double_kernel[bpg, tpb](d_y, d_y)     # 0 transfers
y = d_y.copy_to_host()                # 1 DtoH
```

**The 2025 exam Q15** tests this directly: a kernel called with NumPy arrays `x` and `y` will perform 2 HtoD + 2 DtoH, but only 1 HtoD + 1 DtoH is necessary (since `y` is output-only, it doesn't need to be sent to the GPU first -  allocate it on the GPU with `cuda.device_array` instead).

> KEY RULE: Numba always transfers both input and output arrays when using NumPy. An output-only array doesn't need to be sent to the GPU, so one of the HtoD transfers is wasted. In the optimal version, pre-allocate on the GPU with `cuda.device_array` so there's no HtoD for the output at all.


<div class="alert alert-block alert-danger">

##### **9.10 HtoD/DtoH transfer count questions (exam strategy)**

</div>


**Step 1 - trace the code exactly.** For each array argument, ask:

> Is it input only (read, never written)? → needs HtoD, skip DtoH

> Is it output only (written, never read)? → skip HtoD (allocate with `cuda.device_array`), needs DtoH

> Is it both? → needs both

**Step 2 - is there a loop?** If the function is called N times in a loop, multiply by N unless there is explicit pre-allocation outside the loop.

**Step 3 - what is the theoretical minimum?** The exam sometimes asks "what is optimal". The minimum is always:

> 1 HtoD per unique input (send it once before everything)

> 1 DtoH per unique output (retrieve it once after everything)

**Step 4 - is batching allowed?** This is the ambiguous part. Apply this decision rule:

| Signal in question | Conclusion |
|---|---|
| "arrays are all available upfront" or "same size" | Batching is allowed → 1 HtoD + 1 DtoH is achievable |
| "each array arrives one at a time" or "must return result per iteration" | Batching impossible → 100 HtoD + 1 DtoH is the best you can do |
| Ambiguous / no signal | Pick the answer that pre-allocates outside the loop and copies back once (N HtoD + 1 DtoH) - this is always safe and always better than the naive version |

**The three tiers of transfer count (from worst to best):**

| Approach | HtoD | DtoH | Notes |
|---|---|---|---|
| Naive: numpy arrays inside loop | N | N | Numba auto-transfers both ways every call |
| Pre-allocate outside loop, copy back once | N | 1 | Safe answer when batching is ambiguous |
| Batch all inputs, run once, copy back once | 1 | 1 | Only possible if all data available upfront |

KEY RULE: If the question asks "how many transfers occur" → trace the code literally. If it asks "what is optimal" or "reduces transfers the most" → aim for tier 3 if batching is clearly allowed, otherwise tier 2.

<div class="alert alert-block alert-danger">

##### **9.11 GPU vs CPU benchmark separation (fixed overhead vs per-iteration cost)**

</div>

A common exam pattern: a GPU benchmark looks slower than CPU for a small number of
iterations, and you must decide whether the GPU is actually faster.

**The key insight:** GPU benchmarks have a *fixed* cost (transferring data once over PCIe)
and a *variable* cost (the kernel computation, proportional to iterations). CPU benchmarks
typically have no fixed cost.

**Method - isolate per-iteration cost:**

$$
T_{\text{per iter}}^{\text{GPU}} = \frac{T(N_2) - T(N_1)}{N_2 - N_1}
$$

where $T(N)$ is the total GPU time for $N$ iterations. The fixed transfer cost cancels out.

**Worked example (2025 exam Q23):**

| | 5 iterations | 10 iterations |
|---|---|---|
| CPU (optimised) | 0.5 s | 1.0 s |
| GPU (simple) | 0.85 s | 1.1 s |

CPU per-iteration: $(1.0 - 0.5) / (10 - 5) = 0.1 \text{ s/iter}$

GPU per-iteration: $(1.1 - 0.85) / (10 - 5) = 0.05 \text{ s/iter}$

GPU fixed overhead: $0.85 - 5 \times 0.05 = 0.60 \text{ s}$ (transfer cost)

Break-even point: $0.6 + 0.05n = 0.1n \Rightarrow n = 12$ iterations.

For millions of iterations the GPU is 2 times faster per iteration. The benchmark was
misleading because it measured mostly transfer cost, not compute.

> RULE: Never conclude a GPU implementation is slower based on a benchmark with few iterations. Always separate fixed overhead from per-iteration compute cost. If the GPU per-iteration time is lower, it wins at scale.

<div class="alert alert-block alert-danger">

##### **9.12 Reading nsys profiler output (exam calculation skill)**

</div>


The `nsys` profiler is tested every year. The two core calculations:

**Transfer speed** = total MB / total time:
$$\text{speed (GB/s)} = \frac{\text{Total (MB)}}{\text{Total Time (s)} \times 1024}$$

Or just divide directly: $\frac{25{,}000 \text{ MB}}{2.5 \text{ s}} = 10{,}000 \text{ MB/s} = 10 \text{ GB/s}$.

**Total GPU pipeline time** = kernel time + HtoD time + DtoH time (add all three sections together).

**Which part dominates**: look at `Time (%)` in the `gpumemtimesum` section. If transfers dominate over kernel time, the program is transfer-bound - fix by keeping data GPU-resident. If kernel time dominates, you're actually using the GPU well.

**Worked example (2024 exam Q13–Q14):**

```
** CUDA GPU Kernel Summary:
  100.0   0.5000 s   conv_channels_kernel

** GPU MemOps Summary (by Time):
   83.3   2.5000 s   [CUDA memcpy HtoD]
   16.7   0.5000 s   [CUDA memcpy DtoH]

** GPU MemOps Summary (by Size):
  25000.0 MB   [CUDA memcpy HtoD]
   1000.0 MB   [CUDA memcpy DtoH]
```

Q13 - transfer speed HtoD: $25{,}000 \text{ MB} / 2.5 \text{ s} = 10 \text{ GB/s}$ → answer **(b)**.

Q14 - total GPU time = 0.5 (kernel) + 2.5 (HtoD) + 0.5 (DtoH) = **3.5 s**. CPU takes 7 s. Speedup = 7/3.5 = **2x**. Note: the kernel alone is much faster than the CPU, but transfers eat into the advantage.


From nsys output:

> Fixed overhead = HtoD time + DtoH time (happens once)

> Per-iter cost = kernel time / number of iterations

> Breakeven: overhead / (CPU_per_iter - GPU_per_iter) = N

> ALWAYS check GPU_per_iter < CPU_per_iter first - if not, GPU never wins

<div class="alert alert-block alert-danger">

##### **9.13 Thread block shape and cache performance (exam skill)**

</div>

The block shape affects performance because **warp threads vary along the x-dimension first**. In a 2D block of shape `(Dx, Dy)`, warp 0 contains threads `(0,0), (1,0), ..., (31,0)` - x varies, y is fixed. This means the x-dimension of your block determines which array dimension a warp accesses simultaneously.

For the `matmul_kernel` with inner loop over `k`:

| Block shape | What warp reads | Pattern | Speed |
|---|---|---|---|
| `16 x16` | Mixed row/col | Moderate | ~42 ms |
| `256 x1` | 256 threads in same column of A | Column access → cache misses | ~435 ms (10 x slower) |
| `1 x256` | 256 threads in same row of B | Row access → coalesced | ~29 ms (fastest) |

`1 x256` wins because all warp threads read consecutive elements of the same row of B (`B[k, j], B[k, j+1], ...`), which is perfectly coalesced. This is the same spatial locality lesson from Week 3 - now applied to GPU warps.

**For the 2024R exam Q11–Q12 (average3x3 kernel):**

The kernel accesses `x[row + i, col + j]` - a 3x3 neighbourhood. The question is which block shape makes the warp's memory access coalesced. Since warp threads vary along x (which maps to `col`), a `1 x256` block has all warp threads stepping along consecutive columns in the same row → row-major, coalesced. A `256 x1` block has all warp threads stepping along the same column → cache miss per element.

> Correct answer: **(c) 1 x256** - threads vary along col, giving coalesced row access.

> KEY RULE: Assign the array's **column index** to the block's **x-dimension** so warp threads access consecutive memory addresses (row-major layout = consecutive columns in a row are adjacent in memory).

<div class="alert alert-block alert-danger">

##### **9.14 Reading nsys output - worked calculation**

</div>

`nsys` reports time in nanoseconds. The two things to extract are:
1. **Transfer time** - `cuMemcpyHtoD` (host→device) and `cuMemcpyDtoH` (device→host)
2. **Kernel time** - the CUDA kernel execution itself

**Example nsys stats output (abridged):**

```
CUDA API Statistics:
  Time(%)  Total Time (ns)   Num Calls   Name
  -------  ---------------   ---------   ----
    72.1       368,000,000           2   cuMemcpyHtoD
    24.3       124,000,000           1   cuMemcpyDtoH
     3.6        18,400,000           1   kernel_execution
```

**Reading it:**

> Transfer HtoD: 368 ms total across 2 calls → 184 ms each

> Transfer DtoH: 124 ms

> Kernel: 18.4 ms

> Total: ~510 ms

**Useful-work fraction:** $18.4 / 510 \approx 3.6\%$ - only 3.6% of time is actual
computation. 96.4% is data movement. This tells you the implementation is transfer-bound,
not compute-bound.

**What to do with this information:**

> If transfer dominates: keep data on GPU across calls (`cuda.to_device` once, reuse).
  Use `cuda.device_array` for output so no unnecessary HtoD for output arrays.

> If kernel dominates: optimise the kernel itself (coalescing, shared memory, fewer
  global memory accesses).

**The breakeven calculation:**
If fixed transfer cost is $T_{\text{transfer}}$ and kernel cost per iteration is
$t_{\text{kernel}}$, the GPU breaks even with a CPU running at $t_{\text{CPU}}$ per
iteration when:

$$
n = \frac{T_{\text{transfer}}}{t_{\text{CPU}} - t_{\text{kernel}}}
$$

> EXAM PATTERN: nsys output is given, you must identify whether the bottleneck is
> transfers or compute, and state the fix. The fix is almost always "keep data on GPU"
> when transfers dominate.

<div class="alert alert-block alert-danger">

##### **9.15 Calculating total number of thread blocks (exam calculation)**

</div>

Given output size and threads-per-block, the number of blocks per dimension uses ceiling division:

$$\text{blocks per dim} = \left\lceil \frac{\text{output size}}{\text{tpb}} \right\rceil = \frac{\text{output size} + \text{tpb} - 1}{\text{tpb}}$$



**Example (2025 exam Q13):** output image `200 x200`, block size `16 x 16`:
$$\lceil 200/16 \rceil = \lceil 12.5 \rceil = 13 \quad \Rightarrow \quad 13 \times 13 \text{ blocks}$$

Only the **output array dimensions** matter for counting blocks -  not the size of the input volume. Each thread writes one output element, so you size the grid to cover the output.

<div class="alert alert-block alert-danger">

##### **9.16 GPU overhead vs iteration count trap (exam pattern)**

</div>

A common mistake is benchmarking with too few iterations and concluding the GPU is slower. Total GPU time has a fixed overhead component (memory transfers) plus a per-iteration compute component:

$$T_{\text{GPU}}(N) = T_{\text{overhead}} + N \times T_{\text{kernel}}$$

If $T_{\text{overhead}}$ is large and $N$ is small, the GPU appears slow. With large $N$, the constant overhead amortises and the per-iteration advantage of the GPU dominates.

To isolate the per-iteration cost from two measurements at different $N$:
$$T_{\text{kernel}} = \frac{T(N_2) - T(N_1)}{N_2 - N_1}$$

If the GPU's per-iteration cost is lower than the CPU's, it will win at large enough $N$ even if it looks slower at small $N$.

> KEY EXAM RULE: Always measure GPU performance with enough iterations to amortise the transfer overhead. A benchmark showing "GPU is slower" with 5 iterations may be completely reversed at 500 iterations.

<div class="alert alert-block alert-info">

### **Week 10**

CuPy and GPU Profiling
</div>


<div class="alert alert-block alert-danger">

##### **10.1 GPU latency vs throughput**

</div>


The lecture opens with a fundamental distinction that underpins all GPU programming decisions:

> **CPU targets latency**, time to complete a single task. Fast cores, large caches, branch prediction. Best for sequential logic and tasks that depend on each other.

> **GPU targets throughput**, tasks completed per unit time. Thousands of slow cores, small caches. Best for many independent tasks of the same type.

The numerical example from the slides makes this concrete. For doubling 100 elements:

> CPU (sequential): `100 reads  x 90 ns + 100 writes  x 90 ns + 100 multiplies  x 2 ns` = **18,200 ns**

> GPU (100 parallel threads): `1 read  x 100 ns + 1 multiply  x 40 ns + 1 write  x 100 ns` = **240 ns** (76 times faster)

But for a single element:

> CPU: 90 + 2 + 90 = **182 ns**

> GPU: 100 + 40 + 100 = **240 ns** (CPU wins!)

> KEY INSIGHT: GPUs are not universally faster. They win only when there is enough parallelism to keep all their cores busy. For small or sequential problems the CPU wins on latency.


<div class="alert alert-block alert-danger">

##### **10.2 Shared memory**

</div>

This is the main new hardware concept of Week 10. In Week 9 all kernels read and wrote to **global memory** (the GPU's DRAM). Global memory is slow, similar latency to CPU RAM. Week 10 introduces **shared memory**: a small, fast memory space local to each thread block, roughly L1 cache speed.

**Properties of shared memory:**

> Speed: around L1 cache speed (very fast, $100 \times$ faster than global memory)

> Size: very limited, <100 KB per SM

> Scope: local to one thread block, threads in different blocks cannot see each other's shared memory

> Management: manually controlled by the programmer (unlike L1 cache which is automatic)

> Use case: best when threads in a block access the same data repeatedly, or in access patterns that would cause many global memory cache misses

```python
from numba import cuda
import numpy as np

@cuda.jit
def kernel_with_shared(x, y):
    # Allocate shared memory for this block
    shared = cuda.shared.array(shape=(128,), dtype=np.float32)

    i = cuda.grid(1)
    tid = cuda.threadIdx.x

    # Load from global memory into shared memory
    shared[tid] = x[i]
    cuda.syncthreads()   # wait for ALL threads in block to finish loading

    # Now operate on shared memory (fast)
    y[i] = shared[tid] * 2
```

> CRITICAL: Always call `cuda.syncthreads()` after writing to shared memory and before reading from it. Without it, some threads may read values that other threads have not written yet, a race condition. All threads in the block must reach the `syncthreads()` call or the kernel may hang on older GPUs.


<div class="alert alert-block alert-danger">

##### **10.3 The matrix transpose problem (why shared memory matters)**

</div>

The lecture uses matrix transpose as the motivational example for shared memory. The problem: `y[i, j] = x[j, i]`.

No matter what you do, either reading `x` or writing `y` will be column-wise, and column-wise access on a 2D array is cache-unfriendly (Week 3).

**The shared memory tiling solution:**

```
Step 1: Each thread block loads a BxB tile of x into shared memory -  row-wise (cache friendly read)
Step 2: cuda.syncthreads()
Step 3: Threads write the shared memory tile into y -  transposed -  row-wise (cache friendly write)
```

By going through the shared memory buffer, both the read from `x` and the write to `y` become cache-friendly. This is the "tiling" technique: break the problem into blocks that fit in shared memory, and orchestrate reads/writes to be sequential.


<div class="alert alert-block alert-danger">

##### **10.4 2D warp layout and coalesced access**

</div>

A subtle but important detail from the Week 9/10 slides: in a 2D thread block of size `(Dx, Dy)`, warps are formed by thread **ID**, not by position. The thread ID formula is:

$$\text{ID} = x + y \times D_x$$

This means warp 0 contains threads `(0,0), (1,0), (2,0), ..., (31,0)`, varying $x$, fixed $y$. This has a critical implication for memory access:

```python
# Version A -  i varies with x (warp), j fixed: COLUMN-WISE access → slow
@cuda.jit
def kernel_a(x, y):
    i, j = cuda.grid(2)   # i = row, j = col
    y[i, j] = x[i, j] * 2
# Warp reads x[0,j], x[1,j], ..., x[31,j] -  different rows → not coalesced

# Version B -  j varies with x (warp), i fixed: ROW-WISE access → fast
@cuda.jit
def kernel_b(x, y):
    j, i = cuda.grid(2)   # NOTE: j first, i second
    y[i, j] = x[i, j] * 2
# Warp reads x[i,0], x[i,1], ..., x[i,31] -  same row → coalesced!
```

Timing on a 4096 x4096 array:

> `i, j = cuda.grid(2)` → 0.64 ms (column access, many cache misses)

> `j, i = cuda.grid(2)` → 0.19 ms (row access, coalesced) which is around **$3.4 \times $ faster, same computation**

> KEY INSIGHT: The order we unpack `cuda.grid(2)` determines which dimension varies within a warp. Since warp threads vary along the $x$-dimension first, assign the array's column index to $x$ so warp threads access consecutive columns (row-major = fast).


<div class="alert alert-block alert-danger">

##### **10.5 CuPy: NumPy for the GPU**

</div>

CuPy is a drop-in replacement for NumPy that runs array operations on the GPU. The API is nearly identical to NumPy, just change the import.

```python
import numpy as np
import cupy as cp

# Create on GPU directly
x_gpu = cp.ones((5000, 5000), dtype=cp.float32)

# Transfer between CPU and GPU
x_gpu = cp.asarray(x_cpu)    # NumPy → CuPy (HtoD)
x_cpu = x_gpu.get()           # CuPy → NumPy (DtoH)
x_cpu = cp.asnumpy(x_gpu)     # same as .get()

# Use exactly like NumPy -  runs on GPU
result = 2 * x_gpu             # GPU kernel, no explicit code needed
result = cp.sum(x_gpu, axis=0) # GPU reduction
result = cp.sqrt(x_gpu)        # element-wise

# CuPy arrays can also be passed directly to Numba CUDA kernels
double_kernel[bpg, tpb](x_gpu, y_gpu)
```

**When CuPy is fast vs slow:**

```python
# FAST -  one big vectorised operation, one kernel launch
D = cp.sqrt(cp.sum((p1[:, None, :] - p2[None, :, :]) ** 2, axis=2))

# SLOW -  5000 separate kernel launches inside a Python loop
for i in range(len(p1)):
    row = cp.sqrt(cp.sum((p1[i] - p2) ** 2, axis=1))
    D[i, :] = row
```

The one-loop version causes 5000 separate GPU kernel launches, each doing a small amount of work. The overhead of launching kernels over the PCIe bus dominates. `nsys` will show a massive ratio of `cuLaunchKernel` and memory transfer time vs actual compute.

> The same rule from Week 9 applies: give the GPU **one big problem**, not many small ones.



<div class="alert alert-block alert-danger">

##### **10.6 GPU profiling with `nsys`**

</div>

`nsys` (NVIDIA Nsight Systems) is the standard tool for profiling GPU code. It captures a timeline of CPU and GPU events.

```bash
# Profile and save results
nsys profile -o myprofile python myscript.py

# Show statistics from the profile
nsys stats myprofile.nsys-rep
```


**Key sections to read in `nsys stats` output:**

```
** GPU MemOps Summary (by Time):
Time (%)  Total Time (ns)  Count  Operation
-------- ----------------  -----  ---------
   66.0       3,093,386      1    [CUDA memcpy DtoH]   ← GPU→CPU transfer
   34.0       1,597,044      1    [CUDA memcpy HtoD]   ← CPU→GPU transfer

** GPU MemOps Summary (by Size):
Total (MB)  Count  Operation
----------  -----  ---------
     8.000      1  [CUDA memcpy DtoH]
     8.000      1  [CUDA memcpy HtoD]
```

**D = Device = GPU. H = Host = CPU.**

What to look for:

> **High transfer time, low kernel time** → transfer-bound. Fix: keep data on GPU, use GPU-resident arrays, reduce number of kernel launches.

> **Many `cuLaunchKernel` calls** → Python loop calling GPU repeatedly. Fix: vectorise the whole operation into one call.

> **Low transfer time, high kernel time** → compute-bound. Good, we are actually using the GPU.


**Timing GPU code correctly - `cupyx.time.repeat`:**

```python
from cupyx.time import repeat

# DO NOT use Python's time.perf_counter for GPU timing
# GPU operations are asynchronous -  the CPU returns before GPU finishes

result = repeat(lambda: 2 * my_matrix, n_repeat=200)
# Reports both CPU time and GPU time separately
# CPU: 60 µs  GPU: 785 µs
```

> CRITICAL: Standard Python timing tools (`timeit`, `perf_counter`) do not measure GPU time accurately because GPU operations are asynchronous, the CPU call returns immediately while the GPU is still working. Always use `cupyx.time.repeat` or `cuda.synchronize()` before timing.


<div class="alert alert-block alert-danger">

##### **10.7 Week 10 summary cheat sheet**

</div>


```python
import cupy as cp
from numba import cuda
import numpy as np

# CuPy basics
x_gpu = cp.asarray(x_cpu)       # HtoD
x_cpu = x_gpu.get()             # DtoH
result = cp.sum(x_gpu, axis=0)  # GPU operation

# Shared memory in Numba kernel
@cuda.jit
def kernel(x, y):
    shared = cuda.shared.array(shape=(128,), dtype=np.float32)
    tid = cuda.threadIdx.x
    i = cuda.grid(1)
    shared[tid] = x[i]
    cuda.syncthreads()           # ALWAYS sync after writing shared mem
    y[i] = shared[tid] * 2

# 2D kernel -  coalesced access: j varies with warp's x-dimension
@cuda.jit
def kernel_2d(x, y):
    j, i = cuda.grid(2)          # j=col varies fastest (coalesced)
    if i < x.shape[0] and j < x.shape[1]:
        y[i, j] = x[i, j] * 2

# GPU profiling
# nsys profile -o out python script.py
# nsys stats out.nsys-rep
from cupyx.time import repeat
result = repeat(lambda: cp.sum(x_gpu), n_repeat=200)
```



**GPU memory hierarchy (for exam):**

| Memory | Speed | Scope | Managed by |
|---|---|---|---|
| Registers | Fastest | One thread | Compiler |
| Shared memory | L1 cache | One thread block | Programmer |
| L1/L2 cache | Fast | SM / all SMs | Hardware |
| Global memory (DRAM) | Slow (100 ns) | All threads | Programmer |
| Host memory (CPU RAM) | Very slow (PCIe) | CPU only | CPU |


<div class="alert alert-block alert-danger">

##### **10.8 CuPy one-loop vs no-loop: what nsys actually shows (exam pattern)**

</div>

The one-loop vs no-loop comparison from the exercises is a direct exam topic. The key difference shows up clearly in `nsys`:

**One-loop version** (`for i in range(N): row = cp.sqrt(...)`):

```
** CUDA API Summary:
  65.0%   cuLaunchKernel       ← 135,000+ kernel launches!
  12.8%   cudaMalloc
  12.6%   cudaMemcpyAsync      ← 13,000+ small DtoD copies

** GPU MemOps Summary:
  97.8%   [CUDA memcpy DtoD]   ← 1,467 MB shuffled around inside GPU
```

The `cuLaunchKernel` overhead takes **more time than the sum of all actual kernel runtimes**. Each CuPy operation in the loop is a separate kernel launch, and Python loop overhead + launch overhead dominates completely.

**No-loop version** (one big broadcasting call):

```
** CUDA API Summary:
  (no cuLaunchKernel spam -  only a handful of calls)

** GPU MemOps Summary:
  96.3%   [CUDA memset]        ← one allocation
  2.5%    [CUDA memcpy HtoD]   ← one input transfer
  1.2%    [CUDA memcpy DtoH]   ← one output transfer
```

Each kernel is called only **1–2 times** instead of thousands. The `[CUDA memcpy DtoD]` line is completely gone, meaning no intermediate arrays are being shuffled around the GPU.

> KEY INSIGHT: Unlike NumPy where a Python loop over rows is merely slow, a Python loop over CuPy operations is catastrophic -  each iteration hits the PCIe bus overhead for a kernel launch. Always structure CuPy code as one large operation, not many small ones. This is even more important than for NumPy.

**Speedup observed**: roughly 3x from one-loop to no-loop in CuPy, and around 10x faster than the equivalent NumPy version.


<div class="alert alert-block alert-danger">

##### **10.9 Coalesced memory access in 3D kernels (2025 exam Q12)**

</div>


The warp coalescing principle extends naturally to kernels that access 3D arrays. For a row-wise (C-order) 3D array `vol` with shape `(X, Y, Z)`, the **last axis (Z) has the smallest stride** - consecutive Z indices are adjacent in memory.

Since `j, i = cuda.grid(2)` means `j` (column) varies within a warp, threads access `vol[vi, vj, vk]` where adjacent threads have adjacent `j` values. For coalesced access, `vk` (the Z index) should vary with `j` - meaning `v_step` should point along the Z axis.

**2025 exam Q12 worked example:**

The kernel accesses `vol[vi, vj, vk]` stepping by `v_step` per column (`j`). Options:

| Option | v_step | vk varies with j? | Performance |
|---|---|---|---|
| A | [0, 1, 0] | No -  vj varies, skips rows | **Worst** |
| B | [0, 0, 1] | Yes -  vk varies, consecutive Z | Good |
| C | [0, 0, 1] | Yes -  vk varies, consecutive Z | Good |

Answer: **(A) is worst** because `v_step = [0,1,0]` means adjacent threads access different rows of `vol`, with stride = full Z-plane width. This is exactly the column-access cache miss problem from Week 3, now in 3D.

> KEY RULE: For a row-wise 3D array, the last index (axis 2) is the one that should vary within a warp for coalesced access.

<div class="alert alert-block alert-danger">

##### **10.10 GPU overhead benchmarking trap (2025 exam Q23 - full worked solution)**

</div>

Total GPU runtime has two components:

$$T_{\text{GPU}}(N) = \underbrace{T_{\text{overhead}}}_{\text{constant}} + N \times T_{\text{per iteration}}$$

**Given data:**
> CPU: 0.5 s for 5 iterations → 0.1 s/iteration

> GPU: 0.85 s for 5 iterations, 1.1 s for 10 iterations

**Isolate GPU per-iteration cost:**
$$T_{\text{per iter}} = \frac{1.1 - 0.85}{10 - 5} = \frac{0.25}{5} = 0.05 \text{ s/iter}$$

**Isolate GPU overhead:**
$$T_{\text{overhead}} = 0.85 - 5 \times 0.05 = 0.85 - 0.25 = 0.60 \text{ s}$$

So the GPU takes 0.60 s fixed overhead + 0.05 s per iteration, vs the CPU's 0.1 s per iteration. The GPU is **2 x faster per iteration** but the overhead makes it look slower at low iteration counts.

At $N$ iterations, GPU wins when:
$$0.60 + 0.05N < 0.1N \implies 0.60 < 0.05N \implies N > 12$$

So beyond ~12 iterations the GPU wins, and at millions of iterations the overhead is negligible.

> Correct answer: **(D) The current GPU implementation already has better performance** -  the benchmark at 5–10 iterations just didn't run enough iterations to see it. At real workload (millions of iterations), GPU wins by nearly 2 x.

> KEY EXAM RULE: When given runtimes at two different iteration counts, always compute per-iteration cost as $\Delta T / \Delta N$ before comparing CPU vs GPU. Never compare raw runtimes measured at small N.



<div class="alert alert-block alert-danger">

##### **10.11 GPU job submission on DTU HPC**

</div>


GPU jobs require a different queue and an extra `#BSUB` directive:

```bash
#BSUB -q gpua100                            # GPU queue (not 'hpc'), also choose gpuv100, gpua100, or hpcintogpu
#BSUB -gpu "num=1:mode=exclusive_process"   # request 1 GPU exclusively

# To get an interactive GPU node:
voltash    # V100 GPU
sxm2sh     # A100 SXM2 GPU  
a100sh     # A100 GPU
```

> NOTE: The minimum number of cores for a GPU job is 4. Max wall time on GPU queue is 30 minutes. Always use `#BSUB -gpu "num=1:mode=exclusive_process"` -  `exclusive_process` means no other process shares your GPU, which is important for getting reproducible timings.


<div class="alert alert-block alert-danger">

##### **10.12 `[CUDA memcpy DtoD]` -  what it means**

</div>

A `DtoD` (Device to Device) transfer appears in `nsys` when data is copied within GPU memory -  for example when CuPy creates intermediate arrays during a chain of operations in a loop. It is much faster than HtoD/DtoH (no PCIe involved) but still costs time and indicates unnecessary temporary allocations.

In the one-loop Haversine exercise, `[CUDA memcpy DtoD]` accounts for **1,467 MB** of internal copies across 13,545 operations. In the no-loop version it completely disappears, because the whole computation is fused into a single kernel with no intermediate arrays needing to be passed around.

> If you see `[CUDA memcpy DtoD]` dominating in `nsys`, it means your CuPy code is creating too many intermediate arrays. Fix: restructure into fewer, larger operations.

<div class="alert alert-block alert-danger">

##### **10.13 The nsys sections cheat sheet (exam reading guide)**

</div>

When given `nsys stats` output in the exam, work through these sections in order:

| Section | What to look for |
|---|---|
| `CUDA API Summary` | `cuLaunchKernel` count -  if in the thousands, you have a loop problem |
| `GPU Kernel Summary (gpukernsum)` | Which kernels run, how many times (Instances), total time |
| `GPU MemOps Summary (by Time) (gpumemtimesum)` | Transfer time breakdown -  which dominates? |
| `GPU MemOps Summary (by Size) (gpumemsizesum)` | How many MB transferred -  use for speed calculation |

<div class="alert alert-block alert-danger">

##### **10.14 cProfile output column guide (exam reading)**

</div>


```
ncalls  tottime  percall  cumtime  percall  filename:lineno(function)
```

| Column | Meaning | Exam use |
|---|---|---|
| `ncalls` | Number of calls | High ncalls + small tottime = many cheap calls |
| `tottime` | Time **inside** this function, excluding sub-calls | **Real bottleneck indicator** |
| `percall` (1st) | `tottime / ncalls` | Cost per call, no children |
| `cumtime` | Time including **all sub-calls** | Finds slow top-level functions |
| `percall` (2nd) | `cumtime / ncalls` | Cost per call with children |

**Reading strategy (3 steps):**

1. Sort by `tottime`. The top row is the actual hot spot.
2. If top row is a built-in (`<built-in method ...>`), look at the Python caller just below - your code is calling that built-in in a tight loop.
3. `cumtime >> tottime` means the function just calls other slow functions - drill down.

**Example (2025 exam Q6):**
```
ncalls  tottime  cumtime  filename:lineno(function)
   201    0.001    0.001  render.py:7(load_scene)
   201    0.001    8.841  render.py:13(render_scene)
```
`render_scene` has tiny `tottime` (0.001 s) but huge `cumtime` (8.841 s) → the cost is inside its sub-calls.
The function with the **largest `cumtime`** dominates total runtime from the top-down view.


**Question type 1:** "Where is the bottleneck / what should you optimise?"
→ Use tottime. The function with highest tottime is where CPU time is actually being spent. That's what you pass to line_profiler.

**Question type 2:** "What is dominating total runtime / what takes the most time overall?"
→ Use cumtime. This tells you which high-level function is responsible for most of the wall time, even if it just delegates to sub-calls.

One-line decision rule:

> "What to optimise / where to look next?" → tottime

> "What is slow overall / what dominates?" → cumtime

<div class="alert alert-block alert-danger">

##### **10.15 Memory size unit conversion table (exam reference)**

</div>


| From | To | Operation |
|---|---|---|
| bytes | KB | ÷ 1 024 |
| bytes | MB | ÷ 1 048 576 |
| bytes | GB | ÷ 1 073 741 824 |
| KB | MB | ÷ 1 024 |
| MB | GB | ÷ 1 024 |
| GB | MB | × 1 024 |

**Bytes per element by dtype:**

| dtype | Bytes | Notes |
|---|---|---|
| `bool`, `uint8`, `int8` | 1 | uint8: 0–255; int8: −128–127 |
| `uint16`, `int16`, `float16` | 2 | float16 has very limited range/precision |
| `uint32`, `int32`, `float32` | 4 | |
| `uint64`, `int64`, `float64` | 8 | **NumPy defaults** |
| `complex128` | 16 | two float64 values |

**Array size formula:** $\text{bytes} = \text{n\_elements} \times \text{dtype\_bytes}$

**Example (2025 Q19):** `np.zeros(100000, dtype='uint8')` → $100000 \times 1 = 100000$ bytes $= 97.7$ KB ≈ 100 KB.

**When to downcast (Q17):** Use `uint8` for small non-negative integers (0–255). Never cast numeric data to `datetime`. Avoid `float16` for large ranges - it has only ~3 significant digits and max ~65 504.


When given `nsys stats` output in the exam, work through these sections in order:

| Section | What to look for |
|---|---|
| `CUDA API Summary` | `cuLaunchKernel` count -  if in the thousands, you have a loop problem |
| `GPU Kernel Summary (gpukernsum)` | Which kernels run, how many times (Instances), total time |
| `GPU MemOps Summary (by Time) (gpumemtimesum)` | Transfer time breakdown -  which dominates? |
| `GPU MemOps Summary (by Size) (gpumemsizesum)` | How many MB transferred -  use for speed calculation |

**Transfer speed formula:**
$$\text{speed (GB/s)} = \frac{\text{Total MB}}{\text{Total Time (s)} \times 1024}$$

**Diagnosing the bottleneck:**

| Symptom | Diagnosis | Fix |
|---|---|---|
| `cuLaunchKernel` time > kernel time | Python loop overhead | Vectorise into one big operation |
| HtoD/DtoH >> kernel time | Transfer-bound | Pre-transfer, use GPU-resident arrays |
| `[CUDA memcpy DtoD]` large | Intermediate array churn | Fuse operations, avoid temp arrays |
| Kernel time >> transfers | Compute-bound | Good -  GPU is being used well |

<div class="alert alert-block alert-info">

### **Week 11**

HPC Workflows: Job Arrays and Job Dependencies
</div>

<div class="alert alert-block alert-danger">

##### **11.1 Numba CPU JIT (`@jit` / `@njit`)**

</div>

Numba compiles Python functions to machine code at runtime using LLVM. The key use case is
tight loops over scalar values - exactly where Python is slowest and NumPy can't help.

```python
from numba import njit

@njit
def sum_loop(arr):
    total = 0.0
    for i in range(len(arr)):
        total += arr[i]
    return total
```

`@njit` is shorthand for `@jit(nopython=True)`. Always prefer `@njit`  - it forces Numba to
compile everything to machine code with no fallback to slow Python objects. If it can't
compile, it raises an error immediately rather than silently running slowly.

**When it helps:**

> Loops that iterate over array elements one at a time (can't be vectorised with NumPy)

> Loops with data-dependent branching or early exits

> Recursive algorithms

**When it doesn't help:**

> Already-vectorised NumPy code (no Python loop → nothing to compile)

> Functions that use arbitrary Python objects (dicts, lists of mixed types, classes)

> First call has compilation overhead  - only wins on repeated calls or large inputs

**Exam pattern (2025 Q21):** Given a double for-loop that computes pairwise forces, `@jit`
is a valid speedup because Numba compiles the loop body to machine code. The loop order
matters independently  - Numba does not reorder loops for cache efficiency.

**Limitations Numba cannot handle:**

> Arbitrary Python objects (only NumPy arrays, scalars, tuples of those)

> `try/except` blocks in nopython mode

> Dynamic typing inside the function

> RULE: If you have a loop that can't be replaced by NumPy operations, add `@njit` first
> before reaching for multiprocessing or GPU. It is the cheapest speedup available.

<div class="alert alert-block alert-danger">

##### **11.2 The HPC cluster model**

</div>


The cluster is not just a fast computer, it is a shared resource used by many people simultaneously. A **scheduler** (LSF on DTU's cluster) decides when and where each job runs based on what resources are available. We never run code directly on a compute node; instead you describe what you need in a **job script** and submit it to a queue.

**Job lifecycle:**

```
bsub < script.sh
      ↓
   PEND (waiting for resources)
      ↓
   RUN (executing on compute node)
      ↓
 DONE (success) or EXIT (crash/killed)
```


<div class="alert alert-block alert-danger">

##### **11.3 Job script anatomy (full reference)**

</div>

Every directive starts with `#BSUB`. These look like bash comments but are parsed by `bsub`. The bash script runs after all `#BSUB` lines.

```bash
#!/bin/bash
#BSUB -J myjob           # Job name - shown in bstat/bjobs
#BSUB -q hpc             # Queue name
#BSUB -W 30              # Wall-clock time limit in MINUTES (not seconds) BUT NOTE that the format is in HH:MM, so if we had 02:00 it would be 2 hours but 00:30 OR 30 is 30 minutes, and 2 is minutes but 02:00 is 2 hours, so be careful with the format
#BSUB -n 8               # Number of CPU cores
#BSUB -R "span[hosts=1]" # All cores on the SAME physical node
#BSUB -R "rusage[mem=2GB]" # Memory per CORE (not total!)
#BSUB -o output_%J.out   # stdout file (%J = job ID)
#BSUB -e error_%J.err    # stderr file
#BSUB -B                 # Email when job starts
#BSUB -N                 # Email when job ends (includes resource summary)

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

python myscript.py
```

> CRITICAL MEMORY GOTCHA: `rusage[mem=2GB]` means **2 GB per core**. With `-n 8`, the total reserved memory is `8  x 2 GB = 16 GB`. Always calculate: `total_memory_needed / n_cores` to get the per-core value. Requesting too little causes the job to be killed by the scheduler when it exceeds the reservation.

> Always include `span[hosts=1]` for multiprocessing jobs. Without it, cores may be spread across different physical nodes, and processes that need shared memory (like `mp.RawArray`) will fail silently.

<div class="alert alert-block alert-danger">

##### **11.4 Essential commands**

</div>

```bash
bsub < script.sh          # Submit a job script
bstat                     # Show your running/pending jobs (DTU alias for bjobs)
bjobs                     # Same as bstat
bjobs -A                  # Compact array summary (PEND/RUN/DONE/EXIT counts)
bpeek <JOBID>             # Live stdout of a running job
bpeek <JOBID>[3]          # Live stdout of array element 3
bkill <JOBID>             # Kill a job or entire array
bkill <JOBID>[3]          # Kill only element 3
bjobs -p <JOBID>          # Why is a job still pending?
bjdepinfo <JOBID>         # What does this job depend on (parents)?
bjdepinfo -c <JOBID>      # What depends on this job (children)?
linuxsh                   # Move from login node to interactive work node
```

> ALWAYS run `linuxsh` after SSH login. The login node is shared so running computation there affects every other user. The work node is yours alone.


<div class="alert alert-block alert-danger">

##### **11.5 Job arrays, running many identical jobs at once**

</div>

A job array submits many jobs in one `bsub` call. Each job is identical except for its **index**, stored in `$LSB_JOBINDEX`.

```bash
#!/bin/bash
#BSUB -J subhist[1-203]       # Create 203 jobs, indexed 1 through 203
#BSUB -q hpc
#BSUB -W 15
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -o batch_output/subhist_%J_%I.out   # %I = array index
#BSUB -e batch_output/subhist_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

python huedir.py $LSB_JOBINDEX
```



**Filename placeholders:**

> `%J` - job ID (same for all elements in the array)

> `%I` - array index (unique per element, 1 to N)

> CRITICAL: Always use both `%J` and `%I` in output filenames for arrays. Without `%I`, all 203 jobs write to the same file and their output is interleaved and unreadable.

**Index syntax options:**
```bash
#BSUB -J array[1-100]     # Contiguous: 1, 2, 3, ..., 100
#BSUB -J array[1-100:5]   # Step: 1, 6, 11, 16, ..., 96
#BSUB -J array[2,17,45]   # Explicit list: only these three
```

**Off-by-one warning:** `$LSB_JOBINDEX` starts at 1 (LSF convention), but Python lists are 0-indexed. In your Python script, always convert:
```python
idx = int(sys.argv[1]) - 1   # Convert 1-based LSF index to 0-based Python index
```
Forgetting this means job 1 processes folder 0, job 2 processes folder 1 correctly, but the last folder is never processed.

> Do NOT add `#BSUB -N` email notifications to large job arrays. A 200-element array sends 400 emails on start and end (if we have both -B and -N) so the mail server will reject them.

<div class="alert alert-block alert-danger">

##### **11.6 Job dependencies - chaining jobs together**

</div>

The `-w` directive makes a job wait in PEND state until another job reaches a specified state.

```bash
#BSUB -w job1              # Wait for job named "job1" to reach DONE
#BSUB -w done(job1)        # Explicit: same as above
#BSUB -w done(1234567)     # Wait for specific job ID to reach DONE
#BSUB -w exit(job1)        # Wait for EXIT (crashed or killed)
#BSUB -w ended(job1)       # Wait for DONE or EXIT (any completion)
#BSUB -w started(job1)     # Wait for RUN, EXIT, or DONE (just started)
```

**When to use which:**

> `done()` - use when the next step only makes sense if the previous one succeeded (e.g. aggregation after processing)

> `ended()` - use when the next step should run regardless (e.g. cleanup job)

> `exit()` - rarely used in practice

**Dependency on a job array, wait for all elements:**
```bash
#BSUB -w done(subhist)     # Wait for ALL elements of the "subhist" array
```



**Element-level dependency between two arrays (same length):**
```bash
# This is the NEW job being submitted:
#BSUB -J array2[1-5]
# Each element of array2 waits for the CORRESPONDING element of array1:
#BSUB -w "done(array1[*])"
# → array2[1] waits for array1[1]
# → array2[2] waits for array1[2]  etc.
```
The `[*]` in the dependency refers to array1's indices, matched
one-to-one with array2's indices. Requires both arrays to have the
same length.

<div class="alert alert-block alert-danger">

##### **11.7 The standard HPC workflow pattern - map-reduce**

</div>

The face colors exercise illustrates the canonical pattern for large-scale data processing on a cluster:

```
200,000 images in 203 folders
         ↓
  Job array (subhist[1-203])
  Each job: process one folder → save subhist_<i>.npy
         ↓
  Dependent single job (plothist)
  Waits for done(subhist)
  Aggregates all subhist_*.npy → final histogram
```

This generalises to any embarrassingly parallel problem: split data into chunks, process chunks independently in parallel, aggregate results in a dependent job.

**Map step - `huedir.py`:**
```python
import sys, os
import numpy as np
from PIL import Image

idx = int(sys.argv[1]) - 1   # 1-indexed → 0-indexed
base = '/dtu/projects/02613_2025/data/celeba/images'
folders = sorted(os.listdir(base))
folder = os.path.join(base, folders[idx])

hist = 0
for fname in sorted(os.listdir(folder)):
    img = np.array(Image.open(os.path.join(folder, fname)))
    hsv = np.array(Image.fromarray(img).convert('HSV'))
    hue = hsv[:, :, 0].reshape(-1)
    hist += np.histogram(hue, bins=np.linspace(0, 255, 65))[0]

np.save(f'subhist_{idx}.npy', hist)
```

**Reduce step - `plothuehist.py`:**
```python
from glob import glob
import numpy as np
import matplotlib.pyplot as plt

histfiles = glob('subhist_*.npy')
hist = 0
for f in histfiles:
    hist += np.load(f)
plt.bar(np.linspace(0, 255, 64), hist, width=4)
plt.savefig('histogram.png')
```

**Reduce job script:**
```bash
#!/bin/bash
#BSUB -J plothist
#BSUB -q hpc
#BSUB -W 5
#BSUB -n 1
#BSUB -w done(subhist)       ← waits for all 203 map jobs
#BSUB -R "span[hosts=1]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -o plothist_%J.out
#BSUB -e plothist_%J.err

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613
python plothuehist.py
```

<div class="alert alert-block alert-danger">

##### **11.8 Complete `#BSUB` directive reference**

</div>


| Directive | Meaning |
|---|---|
| `-J name` | Job name |
| `-J name[1-N]` | Job array with indices 1 to N |
| `-q queue` | Queue (`hpc`, `c02613` for GPU) |
| `-W minutes` | Wall-clock time limit |
| `-n cores` | Number of CPU cores |
| `-R "span[hosts=1]"` | All cores on same node |
| `-R "rusage[mem=XGB]"` | Memory **per core** |
| `-o file_%J.out` | stdout file |
| `-e file_%J.err` | stderr file |
| `-o file_%J_%I.out` | stdout for arrays (add `%I`) |
| `-w done(name)` | Dependency: wait for DONE |
| `-w ended(name)` | Dependency: wait for DONE or EXIT |
| `-B` | Email on job start |
| `-N` | Email on job end (with resource summary) |
| `-gpu "num=1"` | Request 1 GPU |


<div class="alert alert-block alert-danger">

##### **11.9 Reading a job script on the exam (critical exam skill)**

</div>



Both the 2024 and 2024R exams open with "what resources does this job script request?" questions. The three things to always read off are **wall time**, **memory**, and **CPU cores** -  and there are two traps in every exam version.

**Trap 1 -  wall time format.** `-W 02:00` is **2 hours**, not 2 minutes. `-W 30` or `-W 00:30` is 30 minutes. The format is `HH:MM` when there's a colon, or bare minutes when there isn't. This is what tripped you on 2024R Q1 -  the answer was 2 hours, not 2 minutes.

**Trap 2 -  memory is per core, not total.** `-R "rusage[mem=4GB]"` with `-n 8` means `8  x 4 GB = 32 GB` total. Always multiply. The exam always gives you a mismatch between what was requested and what was needed, and asks you to fix it:

```bash
# Program needs: 8 cores, 16 GB total, 15 minutes max
# Wrong script:
#BSUB -W 00:05        # Too short (5 min)
#BSUB -R "rusage[mem=4GB]"  # Per-core, but only 1 core → 4 GB total
#BSUB -n 1            # Wrong number of cores

# Corrected:
#BSUB -W 15           # 15 minutes
#BSUB -n 8            # 8 cores
#BSUB -R "rusage[mem=2GB]"  # 2 GB  x 8 cores = 16 GB total ✓
#BSUB -R "span[hosts=1]"    # Keep all cores on same node
```

**2024R Q1 full answer** -  the script had `-W 02:00` (2 hours), `-R "rusage[mem=4GB]"` with `-n 8` (32 GB total), running on a single node. Those are the resources requested.

**2024R Q2** -  to add GPU support:
```bash
#BSUB -q c02613
#BSUB -gpu "num=1:mode=exclusive_process"
```

<div class="alert alert-block alert-danger">

##### **11.10 Reading `bjobs -A` output (exam pattern)**

</div>

`bjobs -A` gives a compact one-line summary per job array:

```
JOBID      ARRAY_SPEC   OWNER    NJOBS  PEND  DONE  RUN  EXIT  ...
21415299   name[1-203]  patmjen  203    195   4     4    1     ...
```

The columns that matter for the exam are `PEND`, `RUN`, `DONE`, and `EXIT`. This is how you diagnose whether a dependent job will ever start:

> `done()` dependency → will only start when **all** elements reach DONE. If any element is in EXIT, the dependency can **never** be satisfied → the downstream job stays PEND forever.

> `ended()` dependency → fires when all elements are in DONE or EXIT (any completion counts).

**2024R Q15 worked example:** 10 `prepare` jobs, 4 DONE, 5 RUN, 1 EXIT. The `compute` job uses `#BSUB -w done(prepare)`. Will `compute` ever start? **No** -  one job has already exited, so the array can never have all 10 in DONE state. If the dependency had been `ended(prepare)`, it would eventually start once the remaining 5 finish running.

> KEY EXAM RULE: Whenever an array element hits EXIT, any downstream `done()` dependency is permanently blocked. This is the most common "will this job ever start?" trap on the exam.

<div class="alert alert-block alert-danger">

##### **11.11 The full job state diagram**

</div>

The LSF state machine has more states than just PEND/RUN/DONE/EXIT:

```
bsub → PEND → (host found) → RUN → (normal completion) → DONE
                                 ↓
                         (abnormal exit / bkill) → EXIT
                         
PEND → (bstop) → PSUSP (user suspended while pending)
RUN  → (bstop) → USUSP (user suspended while running)
RUN  → (host overloaded) → SSUSP (system suspended)
USUSP/SSUSP → (bresume) → RUN
```

For the exam, the key states are PEND, RUN, DONE, EXIT. The SSUSP/USUSP/PSUSP states appear in `bjobs -A` output but you're unlikely to need to reason about them beyond recognising they exist.


<div class="alert alert-block alert-danger">

##### **11.12 `span[hosts=1]` -  when it is required**

</div>

This directive is easy to forget but the exam has caught people on it. It forces all requested cores onto the same physical node. Without it, LSF may spread 8 cores across two different machines. This matters for:

> Any code using `multiprocessing` with shared memory (`mp.RawArray`) -  shared memory is per-machine, so processes on different nodes can't see each other's memory and will crash silently.

> Any code using threading across cores.

> Any code that reads from a local filesystem path that only exists on one node.

Rule of thumb: always include `span[hosts=1]` unless you're explicitly writing distributed-memory code (MPI), which is outside this course.

<div class="alert alert-block alert-danger">

##### **11.13 Exam checklist: writing a correct job script from scratch**

</div>

When asked to write a job script on the exam, work through this list:

1. `#!/bin/bash` -  always first line
2. `-J name` or `-J name[1-N]` for arrays
3. `-q hpc` (or `c02613` for GPU)
4. `-W minutes` -  wall time in correct format
5. `-n cores` -  number of CPU cores
6. `-R "span[hosts=1]"` -  if using multiprocessing or shared memory
7. `-R "rusage[mem=XGB]"` -  **per core**, so `total_GB / n_cores`
8. `-o name_%J.out` and `-e name_%J.err` -  add `%I` for arrays
9. `-w done(jobname)` -  if there is a dependency
10. `-gpu "num=1:mode=exclusive_process"` -  if GPU
11. `source .../conda_init.sh` + `conda activate 02613`
12. The actual command

**Memory calculation reminder:**
$$\text{per-core memory} = \frac{\text{total memory needed}}{\text{number of cores}}$$

So for 16 GB total with 8 cores: `rusage[mem=2GB]`. For 512 MB total with 1 core: `rusage[mem=512MB]`.

<div class="alert alert-block alert-info">

### **Week 12**

Numba CPU (extended notes)
</div>

<div class="alert alert-block alert-danger">

##### **12.1 `@njit` vs `@jit` - what is the difference and which to use**

</div>


`@jit` has two modes:

> **object mode** (`forceobj=True`): Numba falls back to calling the Python interpreter for unsupported constructs. Almost no speedup - sometimes *slower* than plain Python because of overhead.

> **nopython mode** (`nopython=True`): Numba compiles everything to machine code via LLVM with zero Python interpreter involvement. This is where the 100–500× speedups come from.

`@njit` is simply shorthand for `@jit(nopython=True)`. **Always use `@njit`** - if Numba cannot compile the function to machine code it raises an error immediately rather than silently falling back to slow object mode.

```python
from numba import njit, jit

# These are exactly equivalent:
@njit
def f(arr):
    ...

@jit(nopython=True)
def f(arr):
    ...

# AVOID - silently slow if compilation fails:
@jit
def f(arr):
    ...
```

> KEY RULE: `@njit` = fail loudly and fast. `@jit` without `nopython=True` = silently slow. Always use `@njit`.


<div class="alert alert-block alert-danger">

##### **12.2 Compilation is lazy - the warm-up rule**


</div>


Numba compiles the function the **first time it is called**, not when the decorator is applied. The first call is slow (compilation). Every subsequent call is fast (pre-compiled machine code).

```python
@njit
def compute(arr):
    total = 0.0
    for i in range(len(arr)):
        total += arr[i]
    return total

arr = np.ones(1_000_000)

# First call: slow (compiles for dtype float64)
compute(arr)

# Time this - the compiled version
%timeit compute(arr)
```

**Exam trap:** if a benchmark shows `@njit` is slower than pure Python, the most likely reason is that compilation time was included in the measurement. Always warm up before timing.

**Type specialisation:** Numba compiles a separate version for each combination of argument types. Calling `compute` with a `float32` array after `float64` triggers a second compilation. This is automatic and transparent but explains why the *second* call with a new type is also slow.


<div class="alert alert-block alert-danger">

##### **12.3 `cache=True` - persisting compiled code across runs**


</div>


By default, Numba recompiles every time the Python process starts. For long-running HPC jobs this is acceptable. For scripts called repeatedly (e.g. in a job array), the compilation overhead adds up.

```python
@njit(cache=True)
def compute(arr):
    ...
```

With `cache=True`, Numba saves the compiled machine code to a `__pycache__` directory. On the next run, if the function source has not changed, it loads from cache instead of recompiling.

> NOTE: Not needed for exam questions but good practice in actual HPC scripts.

<div class="alert alert-block alert-danger">

##### **12.4 When `@njit` helps and when it does not**

</div>


| Situation | `@njit` helps? | Why |
|---|---|---|
| Tight Python loop over array elements | ✓ YES | Eliminates per-iteration interpreter overhead |
| Loop with data-dependent branching or early exit | ✓ YES | NumPy cannot express this without masks |
| Recursive algorithms | ✓ YES | Python recursion has large interpreter overhead |
| Already-vectorised NumPy code | ✗ NO | No Python loop to compile away; NumPy already calls C |
| Code using dicts, sets, arbitrary Python objects | ✗ NO | Numba cannot infer types → compilation fails |
| `try/except` blocks | ✗ NO | Not supported in nopython mode |
| List/set/dict comprehensions inside the function | ✗ NO | Not supported in nopython mode |
| Generators inside the function | ✗ NO | Not supported in nopython mode |

**The key insight:** `@njit` eliminates the Python interpreter overhead on each loop iteration. If there is no Python loop (vectorised NumPy) or if the loop body uses unsupported Python objects, `@njit` either does nothing or fails.

> EXAM PATTERN (F25 Q21): Given a double for-loop computing pairwise forces, `@jit` is valid because loops are compiled to machine code. But loop *order* matters independently - Numba does not reorder loops for cache efficiency. Swapping loop order can hurt spatial locality even inside `@njit`.


<div class="alert alert-block alert-danger">

##### **12.5 `parallel=True` and `prange` - multi-threaded Numba**

</div>


Numba can parallelise loops across CPU cores with two additions:

1. Add `parallel=True` to the decorator.
2. Replace `range` with `prange` in the outermost loop that is safe to parallelise.

```python
from numba import njit, prange

@njit(parallel=True)
def parallel_sum(arr):
    total = 0.0
    for i in prange(len(arr)):   # prange, not range
        total += arr[i]
    return total
```

**How it works:** Numba compiles the prange loop body to machine code and distributes iterations across threads using OpenMP under the hood. Because the code is GIL-free (no Python interpreter involvement), threads run truly simultaneously - this is the same reason BLAS threading works.

**Typical speedup:** ~3× on 8 cores for the Mandelbrot example in the book. Not linear because of thread startup overhead and the fact that some serial overhead remains.

**When `prange` is safe:** only when loop iterations are **independent** - no iteration writes a value that another iteration reads. The pairwise forces example (F25 Q21) is safe because `forces[i, j]` is only written by one specific `(i, j)` pair.

**When `prange` is NOT safe:** any loop where iteration `i+1` depends on the result of iteration `i` (sequential dependency). Attempting to parallelise such a loop gives wrong results silently.

```python
# UNSAFE - each iteration reads the previous result
@njit(parallel=True)
def cumsum_wrong(arr):
    result = np.zeros(len(arr))
    for i in prange(len(arr)):    # WRONG: i depends on i-1
        result[i] = result[i-1] + arr[i]
    return result
```

> KEY RULE: `prange` = iterations are independent. Use `range` when there are sequential dependencies.


<div class="alert alert-block alert-danger">

##### **12.6 `@vectorize` - turning a scalar function into a NumPy ufunc**

</div>


Numba can wrap a scalar function to operate element-wise across arrays, just like NumPy's built-in universal functions (`np.sin`, `np.exp`, etc.):

```python
from numba import vectorize

@vectorize(["uint8(complex128, uint64)"], target="parallel")
def compute_point(c, max_iter):
    z = 0
    for i in range(max_iter):
        z = z * z + c
        if abs(z) > 2:
            return i
    return max_iter

# Usage - pass arrays, get an array back (no explicit loop)
result = compute_point(positions_array, max_iter)
```

**The type signature** `"uint8(complex128, uint64)"` tells Numba the input and output dtypes. Multiple signatures can be listed for type polymorphism.

**`target="parallel"`** runs the ufunc across multiple threads automatically (same as `prange` but managed by Numba).

**Why `@vectorize` vs `@njit` with `prange`:**

> `@vectorize` is cleaner when the function naturally operates on one element at a time and you want it to work like a NumPy ufunc (broadcasting, output arrays, etc.).

> `@njit` with `prange` is better for more complex loop bodies where you need explicit control over which loop is parallelised.


<div class="alert alert-block alert-danger">

##### **12.7 Numba limitations - what will NOT compile in nopython mode**

</div>



These are the constructs that cause `@njit` to fail. The exam sometimes asks "why can't this be decorated with `@jit(nopython=True)`?":

| Unsupported construct | Workaround |
|---|---|
| Arbitrary Python dicts `{}` | Use two parallel arrays (keys + values) |
| Sets | Use sorted arrays + binary search |
| List/set/dict comprehensions | Pre-allocate a NumPy array, fill with a loop |
| Generators and `yield` | Materialise to a NumPy array first |
| `try / except` blocks | Remove or restructure error handling |
| Nested lists of variable length | Convert to a 2D NumPy array (or ragged array workaround) |
| Calls to arbitrary Python functions | The called function must also be `@njit` decorated |
| Python `print` with f-strings | Use `print(value)` - f-strings are not supported |

**The empty list problem:**
```python
# FAILS - Numba cannot infer the type of an empty list
@njit
def bad(arr):
    result = []           # type unknown!
    for x in arr:
        result.append(x)
    return result

# FIX - pre-allocate a typed array
@njit
def good(arr):
    result = np.empty(len(arr), dtype=arr.dtype)
    for i in range(len(arr)):
        result[i] = arr[i]
    return result
```

> KEY EXAM TRAP (F25 Q22): A function that takes a Python dictionary (`params`) and uses a `while` loop with `params.height > 0` **cannot** be compiled with `@jit(nopython=True)` because:  
> (1) `params` is a Python dict/object - not a supported Numba type  
> (2) The `while` condition is a Python attribute access on an object  
> The fix is to extract scalar values from the dict *before* calling the `@njit` function.


<div class="alert alert-block alert-danger">

##### **12.8 Numba vs NumPy vs multiprocessing - when to reach for each**

</div>

This is the decision flowchart for the exam when asked "how should this code be sped up?":

| Situation | Best tool | Why |
|---|---|---|
| Already vectorised NumPy, bottleneck is memory bandwidth | Nothing / better algorithm | NumPy already calls C; Numba won't help |
| Pure Python loop, can't express with NumPy | `@njit` first | Cheapest speedup, no overhead |
| Independent Python loops, many cores available | `@njit(parallel=True)` + `prange` | Thread-level parallelism, GIL-free |
| Embarrassingly parallel, large data | GPU (`@cuda.jit`) | Thousands of threads |
| I/O-bound tasks | `threading` or `asyncio` | GIL released during I/O |
| CPU-bound Python, GIL is a problem | `multiprocessing.Pool` | Separate processes, own GIL |

GOLDEN RULE: `@njit` before multiprocessing before GPU. It is the cheapest tool and has the lowest overhead. Only escalate when `@njit` is insufficient.

<div class="alert alert-block alert-info">

### **Week 13**

HPC Pitfalls: Common Errors and How to Avoid Them
</div>

<div class="alert alert-block alert-danger">

##### **13.1 Overview**

</div>

Week 13 is the last lecture week, framed as "things that go wrong for real HPC users." The lecture covers five specific pitfalls, each with a concrete symptom, root cause, and fix. The moral is stated explicitly: these are guidelines and warning signs, not hard rules - the exact behaviour depends on the specific cluster setup. But most apply broadly to any Linux HPC system.


<div class="alert alert-block alert-danger">

##### **13.2 Pitfall 1 - Excessive I/O via the `-o/-e` channels**

</div>


**Symptom:** our job runs 6 times slower than expected on the cluster, even though the same code runs fast locally.

**Cause:** the LSF `-o` and `-e` directives route stdout and stderr through a special batch system mechanism that has significant overhead per line written. For programs that print a lot (e.g. a progress bar, per-iteration timing, verbose logging), the I/O overhead completely dominates runtime.

The comparison from the exercise shows this clearly:

| Method | Run time |
|---|---|
| Using `-o`/`-e` channels | 96 s |
| Manual redirect to file | 16 s |

**Fix:** redirect stdout and stderr manually inside the job script using standard bash redirection. Keep `-o` and `-e` in the script to capture the LSF summary (job completion info, resource usage), but redirect the actual Python output separately:

```bash
#!/bin/bash
#BSUB -o name_%J.out     # Captures LSF summary only
#BSUB -e name_%J.err

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

python -u script.py input.npy 4 \
    1> output_${LSB_JOBID}.txt \    # Python stdout → your own file
    2> error_${LSB_JOBID}.txt       # Python stderr → your own file
```

The `-u` flag on `python -u` disables Python's output buffering so output is written immediately rather than held in a buffer (which could cause output to appear truncated if a job crashes).

> NOTE: This speed difference only appears on the `/work3` filesystem on DTU's cluster. On home directories the difference may not be visible. Always use `/work3` for I/O-heavy jobs.


<div class="alert alert-block alert-danger">

##### **13.3 Pitfall 2 - Duplicate I/O with `tee`**

</div>

**Symptom:** disk usage is double what you expected, jobs are slow.

**Cause:** using `tee` in a job script writes output to both a file and stdout simultaneously. Since LSF is already capturing stdout via `-o`, we end up with two copies of every output line, one in our file and one in the LSF output file.

```bash
# BAD -  duplicates all output
python script.py | tee output.txt
# This writes to output.txt AND to the -o channel simultaneously
```

**Fix:** either redirect to a file OR use `-o/-e`, but never both. Since manual redirect is faster (Pitfall 1), prefer:

```bash
python -u script.py > output_${LSB_JOBID}.txt 2>&1
```

> The lecture opens with a real email from HPC support warning about exactly this pattern. The moral: your own code may be fine, but third-party packages or scripts may use `tee` internally without you realising it.


<div class="alert alert-block alert-danger">

##### **13.4 Pitfall 3 - Running on the login node**

</div>

**Symptom:** your code runs, but HPC support contacts you because other users are experiencing slowdowns, or your job gets killed without warning.

**Cause:** the login node is a shared gateway machine -  every user SSHing in lands there. Running any significant computation directly on the login node steals CPU time from everyone else.

**Fix:** always move to a work node with `linuxsh` before running anything, or submit proper batch jobs. This was covered in Week 1 but is repeated here as a pitfall because it affects other users (not just you).


<div class="alert alert-block alert-danger">

##### **13.5 Pitfall 4 - Too many files in one folder**

</div>

**Symptom:** `ls bigdata/` takes 2.6 seconds. File operations on your data directory become sluggish.

**Cause:** most Linux filesystems slow down significantly when a single directory contains thousands of files. `ls` with 100,000 files in one folder takes noticeable time, and every `open()` call on a file in a crowded directory is slower due to the directory index structure.

```bash
$ ls bigdata | wc -l
100000
$ time ls bigdata
...
real    0m2.635s    ← just listing takes 2.6 seconds
```

**Fix:** use subfolders, each containing at most 1000 files. This is exactly the structure used in the CelebA dataset in the course:

```
images/
  000000/   ← ~1000 images
  001000/   ← ~1000 images
  002000/   ← ~1000 images
  ...
```

For Zarr arrays with very many small chunks, use `zarr.NestedDirectoryStore` instead of the default `DirectoryStore`. The nested store organises chunk files into a subdirectory hierarchy, reducing the number of files per directory.

<div class="alert alert-block alert-danger">

##### **13.6 Pitfall 5 - Multi-threading gone wrong**

</div>

This is the most technically rich pitfall and has two opposite failure modes:

**Too few threads - not using available cores:**

```bash
# Request 8 cores in the job script
#BSUB -n 8
# But NumPy uses only 1 thread by default -  7 cores sit idle
python matmuls.py
```

NumPy's linear algebra operations (matmul, dot, SVD, etc.) run through a BLAS backend (OpenBLAS, MKL, etc.) that is multithreaded - but defaults to 1 thread. Simply requesting 8 cores from LSF does nothing unless you also tell BLAS to use them.

**Fix:** set the threading environment variables in the job script before calling Python. Since the exact variable depends on which BLAS implementation is installed, set them all:

```bash
NUM_THREADS=8
OMP_NUM_THREADS=$NUM_THREADS
MPI_NUM_THREADS=$NUM_THREADS
MKL_NUM_THREADS=$NUM_THREADS
OPENBLAS_NUM_THREADS=$NUM_THREADS

python matmuls.py
```

Performance result for a $4000 \times 4000$ matmul:

> Without env vars: 1.72 s (single thread, despite 8 cores requested)

> With env vars set to 8: 0.23 s, around a **$7.4 \times$ speedup**

**Too many threads - thread oversubscription:**

The opposite problem: you are doing parallelism at two levels simultaneously without realising it.

**Example scenario:** 8-thread `ThreadPool` + BLAS with 8 threads = 64 threads competing for 8 cores.

```python
from multiprocessing.pool import ThreadPool

def matmuls(A, B):
    with ThreadPool(8) as p:          # 8 Python threads
        C = np.concatenate(p.starmap(np.matmul, zip(A, B)))
    return C
# Each np.matmul call internally spawns 8 BLAS threads
# → 8  x 8 = 64 concurrent threads on 8 cores
# → OS spends most time context-switching → slower
```

The measured runtimes:

| Configuration | Runtime |
|---|---|
| 1 thread total | 5.87 s |
| BLAS  x 8 only | 1.28 s |
| ThreadPool(8) + BLAS  x 8 | 1.87 s (worse than BLAS alone!) |
| ThreadPool(8) + BLAS  x 1 | 1.33 s (best of both) |

**Fix:** when using a `ThreadPool` for outer-loop parallelism, disable BLAS internal threading so each thread uses exactly 1 BLAS thread:

```bash
OMP_NUM_THREADS=1
MKL_NUM_THREADS=1
OPENBLAS_NUM_THREADS=1
python script.py
```

Or set it in the job script conditionally based on whether you are using `ThreadPool`.

> GOLDEN RULE: total threads $\leq$ physical cores. Any excess causes the OS scheduler to context-switch, wasting CPU cycles. Always count threads from all sources - our code, NumPy/BLAS, and any other libraries.

**Why `ThreadPool` works here despite the GIL:**

This is an important nuance. `ThreadPool` normally cannot bypass the GIL for Python code. But `np.matmul` releases the GIL during the C-level BLAS computation. So while each thread waits inside `np.matmul`, the GIL is released and other threads can genuinely run simultaneously. This is the exception to the Week 5 rule - threading works here specifically because the heavy work happens in C code that explicitly releases the GIL.


<div class="alert alert-block alert-danger">

##### **13.7 Summary of all five pitfalls**

</div>

| # | Pitfall | Symptom | Fix |
|---|---|---|---|
| 1 | Excessive I/O via `-o/-e` | Job much slower than expected | Manually redirect with `1>` and `2>` |
| 2 | Duplicate I/O with `tee` | Double disk usage, slow writes | Choose one output destination |
| 3 | Running on login node | Affects other users, job killed | Always use `linuxsh` or `bsub` |
| 4 | Too many files in one folder | Slow file operations | Use subfolders, around 1000 files each |
| 5a | Too few threads | Cores idle, slow despite `-n N` | Set `OMP_NUM_THREADS` etc. |
| 5b | Too many threads | Slower with more cores | Limit total threads to core count |

<div class="alert alert-block alert-danger">

##### **13.8 The `-u` flag  - why it matters on HPC**

</div>

`python -u` disables Python's output buffering. Without it, Python holds `print()` output in an in-memory buffer and only flushes it to disk in large batches. On HPC this has two consequences:

> `bpeek` shows nothing while the job is running, because output is still sitting in the buffer and hasn't been written yet.

> If the job crashes or is killed, the buffer is lost and you see **no output at all**, making debugging impossible.

With `-u`, every `print()` call flushes immediately. Always use `python -u` in job scripts.

<div class="alert alert-block alert-danger">

##### **13.9 The actual speedup from manual redirect (exercise numbers)**

</div>

The lecture exercise measured with 100,000 lines of output:

| Method | Run time | CPU time |
|---|---|---|
| Using `-o`/`-e` channels | 80–96 s | 7.37 s |
| Manual redirect to `/work3` | 3–16 s | 1.48 s |

The wall time speedup is **up to 26 x** from a one-line change to the job script. The CPU time also drops because fewer time is spent blocked on I/O system calls. Note: the `-o`/`-e` channels still capture the LSF resource summary (the "Successfully completed" block at the end)  - keep those directives, just also add the manual redirect for your program's output.

> CRITICAL DISTINCTION: This effect only appears on `/work3` on DTU HPC. The course dump folder is `/work3/02613/dump/` and is periodically emptied. Always output to `/work3` for I/O-heavy jobs, never to your home directory.

<div class="alert alert-block alert-danger">

##### **13.10 Diagnosing which pitfall you have (decision tree)**

</div>

When a job is slower than expected, work through this checklist:

**Is it slow because of I/O?**
Check: does your program print a lot? Is the run time much larger than CPU time in the LSF summary? → Pitfall 1 or 2. Fix: manual redirect with `1>` and `2>`, remove any `tee`.

**Is it slow because of underused cores?**
Check: you requested `-n 8` but `time python script.py` shows `user ≈ real`. → Pitfall 5a. Fix: set `OMP_NUM_THREADS` etc.

**Is it slow because of too many threads?**
Check: `user >> real` but adding more threads makes it slower. → Pitfall 5b. Fix: set `OMP_NUM_THREADS=1` when using `ThreadPool`.

**Are file operations slow?**
Check: `ls` in your data directory takes seconds. → Pitfall 4. Fix: split into subfolders of ~1000 files.

**Are other users complaining?**
Check: are you running on the login node? → Pitfall 3. Fix: `linuxsh` or `bsub`.